# 03 — Rà soát nhãn synth trước khi sửa

Ngày tạo: 07/10/2026. Mục đích: lập queue các annotation bbox/polygon lệch, tạo file quyết định review và cell nghiệm thu riêng. **Không mặc định thu bbox theo polygon rồi coi lỗi đã sửa.** Owner masks gốc không còn; cần review ảnh/mask hoặc tái sinh với mask chuẩn trước khi chọn biểu diễn sửa.

Protocol chính đã chốt 5 task. Queue dưới đây đọc nguồn synth 4 task để truy về các lỗi hiện hữu; nguồn đó chưa được đưa vào split real-only đã nghiệm thu. Output review ở data/processed/synthetic_annotation_review_v1, không sửa data/archive.

Notebook này hoàn tất **review queue**, chưa hoàn tất **label repair**. Trường decision=pending phải được thay bằng keep_with_justification / regenerate / replace_with_verified_mask và evidence tương ứng; không tự phê duyệt thay người review. Khâu áp sửa chỉ được triển khai khi có mask/config hoặc quyết định cụ thể có thể kiểm chứng.

**Review trực quan:** xuất ảnh đối chiếu toàn bộ 222 case, xem trước case lệch nhất, chọn theo task/annotation ID và ghi quyết định vào CSV. Ảnh gồm toàn cảnh, crop nguyên bản và crop có bbox đỏ/polygon cyan. Gallery/PNG là bằng chứng để kiểm tra, không phải mask đã sửa.


In [ ]:
import torch

# Check if CUDA (GPU support) is available
print("Is GPU available?:", torch.cuda.is_available())

# Get the name of the active GPU
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

Is GPU available?: True
GPU Device Name: NVIDIA GeForce RTX 2050


In [ ]:
import os, sys, json, csv, hashlib, uuid, subprocess, platform, io, base64, html
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(DEVICE)}")

REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists()
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from PIL import Image
from tools.annotation_review import render_review_case, review_crop_box

TZ = ZoneInfo("Asia/Bangkok")


def now():
    return datetime.now(TZ).isoformat(timespec="seconds")


ROOT = Path(
    os.environ.get("AUTOCHECKOUT_DATA_ROOT", str(REPO / "data/archive"))
).resolve()
OUT = REPO / "data/processed/synthetic_annotation_review_v1"
OUT.mkdir(parents=True, exist_ok=True)
run = {
    "run_id": str(uuid.uuid4()),
    "purpose": "visual review of synth mask/bbox discrepancies",
    "started_at": now(),
    "finished_at": None,
    "timezone": "Asia/Bangkok",
    "python": sys.version.split()[0],
    "apply_repairs": False,
}
source_hashes = {}


def dependency_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


run.update(
    {
        "git_commit": subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
        ).strip(),
        "git_dirty": bool(
            subprocess.check_output(
                ["git", "-C", str(REPO), "status", "--porcelain"], text=True
            ).strip()
        ),
        "host": platform.node(),
        "platform": platform.platform(),
        "dependency_versions": {
            name: dependency_version(name)
            for name in ["numpy", "Pillow", "torch", "tqdm"]
        },
        "gpu_used": DEVICE.type == "cuda",
        "compute_device": str(DEVICE),
    }
)
print(json.dumps(run, indent=2))

ARTIFACT = REPO / "runs/data_preprocessing/annotation_review" / run["run_id"]
VISUAL_DIR = ARTIFACT / "visuals"
VISUAL_DIR.mkdir(parents=True, exist_ok=True)
notebook_display_outputs = []


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def show_image(path, label):
    mime = "image/png"
    data = {
        mime: base64.b64encode(Path(path).read_bytes()).decode(),
        "text/plain": label,
    }
    try:
        from IPython.display import display
    except ImportError:
        # Headless execution captures these standard notebook MIME outputs.
        notebook_display_outputs.append(
            {"output_type": "display_data", "metadata": {}, "data": data}
        )
    else:
        display(data, raw=True)

Compute device: cuda
GPU Device Name: NVIDIA GeForce RTX 2050
{
  "run_id": "344b294c-c78c-49c0-a9bd-e7d61e82588d",
  "purpose": "visual review of synth mask/bbox discrepancies",
  "started_at": "2026-10-07T18:48:22+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "python": "3.13.15",
  "apply_repairs": false,
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "git_dirty": true,
  "host": "Laptop",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "dependency_versions": {
    "numpy": "2.5.3",
    "Pillow": "12.3.0",
    "torch": "2.14.0",
    "tqdm": "4.70.1"
  },
  "gpu_used": true,
  "compute_device": "cuda"
}


## 1. Tạo queue độc lập với artifact audit cũ

Quét lại annotations nguồn, ngưỡng 0.98 với tolerance 1e-9. IoU được tính theo batch trên GPU CUDA khi khả dụng; đọc JSON và xuất ảnh vẫn chạy trên CPU. Lưu task/image/annotation ID, class, bbox/polygon bbox, IoU, cutout và checksum nguồn. Không cần file tạm /tmp hoặc run ID cũ để chạy lại notebook.


In [ ]:
queue = []
for t in range(1, 5):
    path = ROOT / f"synth/group/task{t}/annotations.json"
    payload = path.read_bytes()
    source_hashes[str(t)] = hashlib.sha256(payload).hexdigest()
    d = json.loads(payload)
    image_by_id = {i["id"]: i for i in d["images"]}
    parsed = []
    for a in d["annotations"]:
        pts = np.concatenate(
            [np.asarray(p, dtype=float).reshape(-1, 2) for p in a["segmentation"]]
        )
        px, py = pts.min(axis=0)
        pw, ph = pts.max(axis=0) - pts.min(axis=0) + 1
        parsed.append((a, image_by_id[a["image_id"]]["file_name"], (px, py, pw, ph)))

    # Compute all IoUs for a task in one CUDA batch; JSON parsing remains CPU-bound.
    polygon_boxes = torch.tensor(
        [box for _, _, box in parsed], dtype=torch.float64, device=DEVICE
    )
    bboxes = torch.tensor(
        [a["bbox"] for a, _, _ in parsed], dtype=torch.float64, device=DEVICE
    )
    px, py, pw, ph = polygon_boxes.unbind(dim=1)
    x, y, w, h = bboxes.unbind(dim=1)
    inter = torch.clamp(
        torch.minimum(x + w, px + pw) - torch.maximum(x, px), min=0
    ) * torch.clamp(torch.minimum(y + h, py + ph) - torch.maximum(y, py), min=0)
    ious = inter / (w * h + pw * ph - inter + 1e-9)
    for (a, image_file, (px_i, py_i, pw_i, ph_i)), iou in zip(
        parsed, ious.cpu().tolist()
    ):
        if iou < 0.98 - 1e-9:
            queue.append(
                {
                    "source_task": t,
                    "image_id": a["image_id"],
                    "annotation_id": a["id"],
                    "rpc_category_id": a["category_id"],
                    "image_file": image_file,
                    "cutout": a["cutout"],
                    "bbox": a["bbox"],
                    "polygon_bbox": [
                        float(px_i),
                        float(py_i),
                        float(pw_i),
                        float(ph_i),
                    ],
                    "iou": float(iou),
                    "source_sha256": source_hashes[str(t)],
                    "decision": "pending",
                    "evidence": "",
                    "reviewer": "",
                }
            )
    del d, payload, parsed, polygon_boxes, bboxes, ious
queue.sort(key=lambda r: r["iou"])
JSON_PATH = OUT / "review_queue.json"
CSV_PATH = OUT / "review_decisions.csv"
# Preserve user decisions on rerun; never reset a filled review file.
if JSON_PATH.exists():
    previous = json.loads(JSON_PATH.read_text())
    assert {
        (r["source_task"], r["annotation_id"], r["source_sha256"]) for r in previous
    } == {(r["source_task"], r["annotation_id"], r["source_sha256"]) for r in queue}
else:
    JSON_PATH.write_text(json.dumps(queue, ensure_ascii=False, indent=2))
if not CSV_PATH.exists():
    with CSV_PATH.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "source_task",
                "annotation_id",
                "decision",
                "evidence",
                "reviewer",
            ],
        )
        writer.writeheader()
        for r in queue:
            writer.writerow({k: r[k] for k in writer.fieldnames})
print("Review cases:", len(queue), "| min IoU:", min(r["iou"] for r in queue))
print("Decisions CSV:", str(CSV_PATH.relative_to(REPO)))

decisions_before_sha256 = hash_file(CSV_PATH)

# Ensure queued annotations are back on CPU-native Python objects for rendering cells.

Review cases: 222 | min IoU: 0.7266213202222918
Decisions CSV: data/processed/synthetic_annotation_review_v1/review_decisions.csv


## 2. Xuất ảnh đối chiếu đủ 222 case

Mỗi PNG có ba phần: **toàn cảnh khay**, **crop nguyên bản**, **cùng crop có bbox đỏ và biên polygon cyan**. Crop lấy hợp của bbox và mọi mảnh polygon, thêm 32 px rồi giới hạn trong ảnh; hai crop giữ cùng tỉ lệ để đối chiếu. Không chỉnh geometry hoặc áp sửa nhãn.

Xuất đầy đủ PNG và gallery HTML theo run ID ở `runs/data_preprocessing/annotation_review/`. Xếp IoU tăng dần, hiện trước 3 case lệch nhất trong notebook. Gallery là snapshot hình ảnh; quyết định review được quản lý trong CSV, không suy ra từ việc PNG đã được xuất.


In [4]:
annotations_for_review = {}
for t in range(1, 5):
    d = json.loads((ROOT / f"synth/group/task{t}/annotations.json").read_text())
    wanted = {r["annotation_id"] for r in queue if r["source_task"] == t}
    annotations_for_review.update(
        {(t, a["id"]): a for a in d["annotations"] if a["id"] in wanted}
    )
    del d
assert len(annotations_for_review) == len(queue)
source_assets = {ROOT / r[k] for r in queue for k in ["image_file", "cutout"]}
asset_hashes_before = {
    str(p.relative_to(ROOT)): hash_file(p) for p in sorted(source_assets)
}
visual_records = []
for rank, case in enumerate(queue, 1):
    key = (case["source_task"], case["annotation_id"])
    ann = annotations_for_review[key]
    assert (
        ann["id"],
        ann["image_id"],
        ann["category_id"],
        ann["bbox"],
        ann["cutout"],
    ) == (
        case["annotation_id"],
        case["image_id"],
        case["rpc_category_id"],
        case["bbox"],
        case["cutout"],
    )
    path = VISUAL_DIR / f"task{key[0]}_ann{key[1]}.png"
    with Image.open(ROOT / case["image_file"]) as image:
        crop_box = review_crop_box(ann["bbox"], ann["segmentation"], image.size)
        render_review_case(image, ann, case).save(path)
    visual_records.append(
        {
            "rank": rank,
            "source_task": key[0],
            "annotation_id": key[1],
            "image_id": case["image_id"],
            "rpc_category_id": case["rpc_category_id"],
            "iou": case["iou"],
            "image_file": case["image_file"],
            "cutout": case["cutout"],
            "crop_box": list(crop_box),
            "preview": str(path.relative_to(REPO)),
            "preview_sha256": hash_file(path),
            "source_sha256": case["source_sha256"],
        }
    )
(ARTIFACT / "visual_manifest.json").write_text(
    json.dumps(visual_records, ensure_ascii=False, indent=2)
)
(ARTIFACT / "source_asset_sha256.json").write_text(
    json.dumps(asset_hashes_before, indent=2)
)
cards = []
for r in visual_records:
    filename = Path(r["preview"]).name
    label = f"#{r['rank']} | task {r['source_task']} / ann {r['annotation_id']} / SKU {r['rpc_category_id']} | IoU {r['iou']:.4f}"
    cards.append(
        f'<article><a href="{html.escape(filename)}"><img loading="lazy" src="{html.escape(filename)}" alt="{html.escape(label)}"></a><p>{html.escape(label)}</p></article>'
    )
GALLERY = VISUAL_DIR / "index.html"
GALLERY.write_text(
    '<!doctype html><html lang="vi"><meta charset="utf-8"><title>RPC annotation review</title>'
    "<style>body{font:16px system-ui;margin:24px;background:#f5f5f5;color:#222}"
    "main{display:grid;grid-template-columns:repeat(auto-fit,minmax(480px,1fr));gap:20px}"
    "article{background:white;padding:12px}img{width:100%}p{margin:8px 0}"
    "@media(max-width:520px){main{grid-template-columns:1fr}}</style>"
    "<h1>222 case cần review</h1><p>Đỏ: bbox đã lưu. Cyan: biên polygon đã lưu. "
    "Bấm ảnh để xem kích thước đầy đủ. PNG đã xuất không đồng nghĩa nhãn đã được nghiệm thu.</p>"
    "<p>Run "
    + html.escape(run["run_id"])
    + "</p><main>"
    + "".join(cards)
    + "</main></html>"
)
print(
    "Visual exports:",
    len(visual_records),
    "| cases IoU < 0.9:",
    sum(r["iou"] < 0.9 for r in queue),
)
print("Gallery:", str(GALLERY.relative_to(REPO)))
print("Manifest:", str((ARTIFACT / "visual_manifest.json").relative_to(REPO)))
PREVIEW_COUNT = 3
for r in visual_records[:PREVIEW_COUNT]:
    show_image(
        REPO / r["preview"],
        f"Task {r['source_task']} / ann {r['annotation_id']} / IoU {r['iou']:.4f}",
    )

Visual exports: 222 | cases IoU < 0.9: 52
Gallery: runs/data_preprocessing/annotation_review/344b294c-c78c-49c0-a9bd-e7d61e82588d/visuals/index.html
Manifest: runs/data_preprocessing/annotation_review/344b294c-c78c-49c0-a9bd-e7d61e82588d/visual_manifest.json


Preview retained in local notebook snapshot: runs/notebook_versions/03_annotation_review/before-docs-cleanup-20261008T143902+0700.ipynb


Preview retained in local notebook snapshot: runs/notebook_versions/03_annotation_review/before-docs-cleanup-20261008T143902+0700.ipynb


Preview retained in local notebook snapshot: runs/notebook_versions/03_annotation_review/before-docs-cleanup-20261008T143902+0700.ipynb


## 3. Chọn một case để review

Đặt `CASE_KEY = (source_task, annotation_id)` rồi chạy lại cell bên dưới; ví dụ `(4, 15271)`. `None` chọn case có IoU thấp nhất. ID task ở đây là **task nguồn 4-task**, dùng truy vết annotation cũ, không phải mapping 5 task mới.

Cell hiện lại ảnh đối chiếu, cutout nguồn và quyết định hiện tại trong CSV. Cutout giúp nhận diện vật thể; nó chưa thay thế owner-mask sau biến đổi/che khuất trong ảnh ghép. Mỗi lần đổi case chỉ cần chạy cell này, không cần tạo lại toàn bộ gallery.


In [1]:
CASE_KEY = None  # Ví dụ: (4, 15271). None = case lệch nhiều nhất.
case_key = CASE_KEY or (queue[0]["source_task"], queue[0]["annotation_id"])
record_by_key = {(r["source_task"], r["annotation_id"]): r for r in visual_records}
if case_key not in record_by_key:
    raise ValueError(
        f"Case {case_key} không nằm trong review queue; kiểm tra task/annotation ID"
    )
selected = record_by_key[case_key]
with CSV_PATH.open(newline="") as f:
    current_decisions = list(csv.DictReader(f))
current = next(
    r
    for r in current_decisions
    if (int(r["source_task"]), int(r["annotation_id"])) == case_key
)
print(json.dumps({"case": selected, "decision": current}, ensure_ascii=False, indent=2))
show_image(
    REPO / selected["preview"], f"Case {case_key}: full tray / original crop / labels"
)
cutout_path = ARTIFACT / f"cutout_task{case_key[0]}_ann{case_key[1]}.png"
with Image.open(ROOT / selected["cutout"]) as cutout:
    thumb = cutout.copy()
    thumb.thumbnail((360, 360))
    thumb.save(cutout_path)
print("Cutout nguồn (chưa biến đổi theo ảnh ghép):", selected["cutout"])
show_image(cutout_path, f"Original cutout for case {case_key}")
# Read AI suggestions separately from the manual CSV.
AI_LATEST=REPO/'data/processed/synthetic_annotation_review_v1/ai/gpt-6-luna_v1/latest_summary.json'
if AI_LATEST.exists():
    ai_latest=json.loads(AI_LATEST.read_text())
    ai_saved=json.loads((REPO/ai_latest['results']).read_text())
    ai_case=next((r for r in ai_saved if tuple(r['case_key'])==case_key and r['source_sha256']==selected['source_sha256']),None)
    if ai_case:
        print('ĐỀ XUẤT AI — chưa áp sửa / chưa thay quyết định manual:')
        print(json.dumps({'model':ai_case['model'],'prompt_version':ai_case['prompt_version'],
                          'review':ai_case['review'],'response_id':ai_case['response_id'],
                          'pilot_quality_pass':ai_latest['pilot_quality_pass']},ensure_ascii=False,indent=2))
    else:print('Case này chưa có đề xuất trong đợt AI gần nhất.')


{
  "case": {
    "rank": 1,
    "source_task": 4,
    "annotation_id": 15271,
    "image_id": 1240,
    "rpc_category_id": 196,
    "iou": 0.7266213202222918,
    "image_file": "synth/group/task4/images/001240.jpg",
    "cutout": "cutouts/sam/objects/196_stationery/6951384903194~B_camera2-14.png",
    "crop_box": [
      907,
      307,
      1225,
      1010
    ],
    "preview": "runs/data_preprocessing/annotation_review/520cad15-9b84-452f-809c-b3ee23c6c02a/visuals/task4_ann15271.png",
    "preview_sha256": "420514c227f08a4d77721702fca0d2a8d4d34880f6bdaa5e581e6e955e93d65b",
    "source_sha256": "907dd3b9be7241a2d3b5761f50fde9eb7be195a159f2005c5a3175f4871653d6"
  },
  "decision": {
    "source_task": "4",
    "annotation_id": "15271",
    "decision": "pending",
    "evidence": "",
    "reviewer": ""
  }
}
Cutout nguồn (chưa biến đổi theo ảnh ghép): cutouts/sam/objects/196_stationery/6951384903194~B_camera2-14.png
ĐỀ XUẤT AI — chưa áp sửa / chưa thay quyết định manual:
{
  "model": "g

Preview retained in local notebook snapshot: runs/notebook_versions/03_annotation_review/before-docs-cleanup-20261008T143902+0700.ipynb


Preview retained in local notebook snapshot: runs/notebook_versions/03_annotation_review/before-docs-cleanup-20261008T143902+0700.ipynb


## 4. Ghi quyết định sau khi kiểm tra

Mở `data/processed/synthetic_annotation_review_v1/review_decisions.csv`, tìm đúng cặp `source_task,annotation_id` và sửa ba trường `decision,evidence,reviewer`. Giữ nguyên IDs và các dòng khác; nếu evidence chứa dấu phẩy phải đặt trong dấu ngoặc kép theo CSV. Chạy lại cell chọn case để đọc quyết định mới, rồi chạy cell nghiệm thu cuối để cập nhật số đã review. Notebook giữ nguyên CSV khi chạy lại.

| decision                     | Khi dùng                               | Evidence cần có                                                                              |
| ---------------------------- | -------------------------------------- | -------------------------------------------------------------------------------------------- |
| `pending`                    | Chưa đủ bằng chứng                     | Để trống hoặc ghi câu hỏi còn thiếu                                                          |
| `keep_with_justification`    | Có căn cứ giữ bbox cho detector        | Đường dẫn PNG + nhận xét cụ thể phần vật nằm ngoài polygon; giải trình giới hạn segmentation |
| `regenerate`                 | Không xác định được mask/geometry đúng | Đường dẫn PNG + mô tả vì sao cần tái sinh; config/mask/provenance bổ sung sau                |
| `replace_with_verified_mask` | Có mask sau ghép đã xác minh           | Đường dẫn mask/bằng chứng và nguồn tạo mask; chưa áp sửa chỉ vì đã chọn quyết định này       |

Ví dụ evidence: `runs/data_preprocessing/annotation_review/<run_id>/visuals/task4_ann15271.png; <nhận xét sau khi xem ảnh>`. Chỉ điền kết luận sau khi kiểm tra. Chọn `regenerate` hoặc `replace_with_verified_mask` là quyết định công việc, chưa chứng minh việc sửa/tái sinh đã hoàn tất.


## 5. Cell nghiệm thu visual review và trạng thái repair

Gate queue: keys duy nhất, ảnh/cutout có thật, checksum nguồn không đổi, đúng 222 lỗi theo snapshot đã audit. Gate sửa nhãn: mọi case có decision/evidence/reviewer, mask thay thế hoặc regeneration có provenance, bbox/area/segmentation cùng một mask cuối, các JSON phụ thuộc được cập nhật và split hash leakage được kiểm tra lại. Chưa có verified masks/decisions nên gate repair phải ở pending.

Nghiệm thu ảnh đối chiếu: đủ 222 PNG theo queue, đúng IDs/geometry, giải mã được từng PNG, checksum ảnh/cutout/annotation nguồn giữ nguyên. Số PNG xuất thành công tách riêng với số quyết định review có evidence.


In [ ]:
assert len({(r["source_task"], r["annotation_id"]) for r in queue}) == len(queue)
assert (
    len(queue) == 222
), "Dataset changed: rerun acceptance audit, do not assume old issue count"
assert all(
    (ROOT / r["image_file"]).is_file() and (ROOT / r["cutout"]).is_file() for r in queue
)
for t, h in source_hashes.items():
    assert (
        hashlib.sha256(
            (ROOT / f"synth/group/task{t}/annotations.json").read_bytes()
        ).hexdigest()
        == h
    )
with CSV_PATH.open(newline="") as f:
    decisions = list(csv.DictReader(f))
assert {(int(r["source_task"]), int(r["annotation_id"])) for r in decisions} == {
    (r["source_task"], r["annotation_id"]) for r in queue
}
allowed_decisions = {
    "pending",
    "keep_with_justification",
    "regenerate",
    "replace_with_verified_mask",
}
assert len(decisions) == len(queue) and all(
    r["decision"] in allowed_decisions for r in decisions
)
ready_decisions = [
    r
    for r in decisions
    if r["decision"] != "pending" and r["evidence"].strip() and r["reviewer"].strip()
]
assert len(visual_records) == len(queue)
assert {(r["source_task"], r["annotation_id"]) for r in visual_records} == {
    (r["source_task"], r["annotation_id"]) for r in queue
}
for r in visual_records:
    path = REPO / r["preview"]
    assert hash_file(path) == r["preview_sha256"]
    with Image.open(path) as image:
        image.load()
        assert image.size == (1168, 720)
assert {
    name: hash_file(ROOT / name) for name in asset_hashes_before
} == asset_hashes_before
visual_acceptance = "PASS"

run.update({"finished_at": now(), "status": "VISUAL_REVIEW_READY_REPAIR_PENDING"})
run["duration_seconds"] = (
    datetime.fromisoformat(run["finished_at"])
    - datetime.fromisoformat(run["started_at"])
).total_seconds()
summary = {
    "run": run,
    "queue_cases": len(queue),
    "reviewed_cases_with_evidence": len(ready_decisions),
    "queue_acceptance": "PASS",
    "repair_acceptance": "PENDING_VERIFIED_MASK_OR_REGENERATION",
    "source_sha256": source_hashes,
    "source_modified": False,
    "visual_acceptance": visual_acceptance,
    "visual_cases_exported": len(visual_records),
    "visual_manifest": str((ARTIFACT / "visual_manifest.json").relative_to(REPO)),
    "gallery": str(GALLERY.relative_to(REPO)),
    "decisions_before_sha256": decisions_before_sha256,
    "decisions_after_sha256": hash_file(CSV_PATH),
    "required_before_synth_training": [
        "complete verified review/repair",
        "rebuild dependent JSONs",
        "5-task mapping",
        "generation provenance",
        "cross-split checks",
    ],
}
(OUT / "review_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)
)
(ARTIFACT / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)
)
print(json.dumps(summary, indent=2))

NameError: name 'queue' is not defined

## 6. Review bằng OpenAI — pilot 10 case trước

Model `gpt-6-luna`, Responses API + structured output, reasoning `low` (sau pilot `none` chưa đạt), ảnh `high`. Đọc key từ biến môi trường `OPENAI_API_KEY` hoặc `.env` ở root; không ghi key vào notebook/config/output. Dùng thư viện chuẩn Python, không cần cài SDK hoặc dotenv.

Chạy 10 case có đủ task nguồn và mức lệch nặng/gần ngưỡng trước; kiểm tra nhận xét với ảnh rồi lưu `pilot_quality_audit.json`. Chỉ chạy phần còn lại khi audit nội dung này đạt và đúng fingerprints của pilot. JSON hợp lệ chưa đủ để coi pilot đạt chất lượng.

Các cell AI bên dưới có thể chạy độc lập với phần xuất gallery, dùng artifacts visual đã nghiệm thu. Đặt `AI_RUN_PILOT=True` trong cell cấu hình để gọi API; chạy lại cùng input dùng cache. Đặt `AI_RUN_REMAINING=True` sau khi đã có audit pilot đạt. Mặc định hai cờ tắt để Run All phục vụ audit dữ liệu không tự tạo chi phí.

Kết quả lưu ở `data/processed/synthetic_annotation_review_v1/ai/gpt-6-luna_v1/` và log theo run ID trong `runs/data_preprocessing/ai_review/`. Đây là **đề xuất AI**, tách riêng CSV người review và nhãn đã nghiệm thu. Model không được đề xuất mask đã xác minh khi mask đó chưa tồn tại. Snapshot config: [annotation_review_ai_v1.json](../../configs/data/annotation_review_ai_v1.json). [OpenAI Docs](https://developers.openai.com/api/docs/models/gpt-6-luna), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs).


In [1]:
import os,sys,json,uuid,subprocess,platform,csv
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
AI_REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
if str(AI_REPO) not in sys.path:sys.path.insert(0,str(AI_REPO))
from tools.ai_annotation_review import (MODEL,PROMPT_VERSION,PROMPT,select_pilot,load_api_key,
    create_response,review_case,pilot_gate_passed,file_sha256,write_json,timestamp)
AI_RUN_PILOT=os.getenv('AUTOCHECKOUT_REVIEW_AI_PILOT','0')=='1'  # Có thể đổi thành True.
AI_RUN_REMAINING=os.getenv('AUTOCHECKOUT_REVIEW_AI_REMAINING','0')=='1'  # Sau audit pilot đạt.
AI_ROOT=Path(os.getenv('AUTOCHECKOUT_DATA_ROOT',str(AI_REPO/'data/archive'))).resolve()
AI_REVIEW_ROOT=AI_REPO/'data/processed/synthetic_annotation_review_v1'
AI_VISUAL_SUMMARY=json.loads((AI_REVIEW_ROOT/'review_summary.json').read_text())
assert AI_VISUAL_SUMMARY['visual_acceptance']=='PASS'
AI_CASES=json.loads((AI_REPO/AI_VISUAL_SUMMARY['visual_manifest']).read_text())
assert len(AI_CASES)==222
ai_queue_by_key={(r['source_task'],r['annotation_id']):r for r in json.loads((AI_REVIEW_ROOT/'review_queue.json').read_text())}
for r in AI_CASES:
    q=ai_queue_by_key[r['source_task'],r['annotation_id']]
    assert q['source_sha256']==r['source_sha256']
    r.update({k:q[k] for k in ['bbox','polygon_bbox']})
for task,digest in AI_VISUAL_SUMMARY['source_sha256'].items():
    assert file_sha256(AI_ROOT/f'synth/group/task{task}/annotations.json')==digest
AI_PILOT=select_pilot(AI_CASES,count=10)
AI_OUT=AI_REVIEW_ROOT/'ai/gpt-6-luna_v1';AI_OUT.mkdir(parents=True,exist_ok=True)
AI_ARTIFACT=AI_REPO/'runs/data_preprocessing/ai_review'/str(uuid.uuid4());AI_ARTIFACT.mkdir(parents=True,exist_ok=True)
AI_MANUAL_CSV=AI_REVIEW_ROOT/'review_decisions.csv';AI_MANUAL_HASH=file_sha256(AI_MANUAL_CSV)
AI_SETTINGS=json.loads((AI_REPO/'configs/data/annotation_review_ai_v1.json').read_text())
assert AI_SETTINGS['model']==MODEL and AI_SETTINGS['prompt_version']==PROMPT_VERSION
ai_run={'run_id':AI_ARTIFACT.name,'purpose':'OpenAI pilot 10 then conditional remaining reviews',
        'started_at':timestamp(),'finished_at':None,'timezone':'Asia/Bangkok','python':sys.version.split()[0],
        'host':platform.node(),'git_commit':subprocess.check_output(['git','-C',str(AI_REPO),'rev-parse','HEAD'],text=True).strip(),
        'git_dirty':bool(subprocess.check_output(['git','-C',str(AI_REPO),'status','--porcelain'],text=True).strip()),
        'settings':AI_SETTINGS,'code_sha256':file_sha256(AI_REPO/'tools/ai_annotation_review.py'),
        'visual_input_manifest':AI_VISUAL_SUMMARY['visual_manifest'],'source_sha256':AI_VISUAL_SUMMARY['source_sha256'],
        'pilot_enabled':AI_RUN_PILOT,'remaining_enabled':AI_RUN_REMAINING,'status':'RUNNING'}
write_json(AI_ARTIFACT/'run.json',ai_run);write_json(AI_ARTIFACT/'pilot_selection.json',AI_PILOT)
(AI_ARTIFACT/'prompt.txt').write_text(PROMPT)
AI_KEY=load_api_key(AI_REPO/'.env') if AI_RUN_PILOT or AI_RUN_REMAINING else None
ai_pilot_results=[];ai_remaining_results=[]
print(json.dumps({**ai_run,'pilot_case_keys':[(r['source_task'],r['annotation_id']) for r in AI_PILOT]},ensure_ascii=False,indent=2))


{
  "run_id": "10897daf-1644-4404-873f-d485ba24bda0",
  "purpose": "OpenAI pilot 10 then conditional remaining reviews",
  "started_at": "2026-10-07T21:01:13+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "python": "3.14.4",
  "host": "Laptop",
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "git_dirty": true,
  "settings": {
    "model": "gpt-6-luna",
    "prompt_version": "rpc-bbox-polygon-review-v3",
    "endpoint": "https://api.openai.com/v1/responses",
    "reasoning_effort": "low",
    "image_detail": "high",
    "cutout_max_side": 512,
    "max_output_tokens": 3000,
    "pilot_cases": 10,
    "workers": 4,
    "require_visual_pilot_audit": true,
    "apply_label_repairs": false,
    "write_manual_decisions": false,
    "pricing_checked_on": "2026-10-07",
    "pricing_mode": "standard_short_context",
    "usd_per_million_tokens": {
      "input": 0.1,
      "cached_input": 0.01,
      "cache_write": 0.125,
      "output": 0.5
    },
    "documentatio

## 7. Gọi API cho 10 case pilot

Mỗi request gửi ảnh đối chiếu và cutout nguồn được thu tối đa 512 px/đặt trên nền trắng. ID, IoU và SKU là metadata; model nhận xét pixels, không tự sửa geometry. Cache phụ thuộc nội dung ảnh/prompt/config/source hash, không phụ thuộc đường dẫn run mới. Ghi receipt API, usage và thời điểm trước khi phân tích JSON để giữ bằng chứng khi response chưa hoàn chỉnh.

Pilot synchronous để đọc kết quả ngay. Giá Standard thực tính từ usage được trả về; số USD là ước tính theo bảng giá, không phải hóa đơn. Xem cả lý do/observations và ảnh trước khi đánh giá chất lượng.


In [2]:
if AI_RUN_PILOT or AI_RUN_REMAINING:
    def ai_send(payload):return create_response(payload,AI_KEY)
    def ai_review_one(case):
        assert file_sha256(AI_REPO/case['preview'])==case['preview_sha256']
        return review_case(case,repo=AI_REPO,raw_root=AI_ROOT,out_dir=AI_OUT,send=ai_send)
    with ThreadPoolExecutor(max_workers=2) as pool:ai_pilot_results=list(pool.map(ai_review_one,AI_PILOT))
    assert len(ai_pilot_results)==10
    write_json(AI_ARTIFACT/'pilot_results.json',ai_pilot_results)
    write_json(AI_OUT/'pilot_results.json',ai_pilot_results)
    pilot_summary={'cases':len(ai_pilot_results),'cached':sum(r['cached'] for r in ai_pilot_results),
                   'decision_counts':dict(Counter(r['review']['decision'] for r in ai_pilot_results)),
                   'estimated_total_cost_usd':sum(r['estimated_cost_usd'] for r in ai_pilot_results),
                   'quality_gate':'PENDING_VISUAL_AUDIT','manual_csv_modified':False,'repairs_applied':False}
    write_json(AI_ARTIFACT/'pilot_summary.json',pilot_summary)
    print(json.dumps(pilot_summary,ensure_ascii=False,indent=2))
    for r in ai_pilot_results:
        print(json.dumps({'case_key':r['case_key'],'review':r['review'],'usage':r['usage'],
                          'estimated_cost_usd':r['estimated_cost_usd']},ensure_ascii=False))
else:print('API pilot disabled. Set AI_RUN_PILOT=True in the setup cell, then execute setup and this cell.')


{
  "cases": 10,
  "cached": 10,
  "decision_counts": {
    "regenerate": 6,
    "keep_with_justification": 3,
    "pending": 1
  },
  "estimated_total_cost_usd": 0.00605975,
  "quality_gate": "PENDING_VISUAL_AUDIT",
  "manual_csv_modified": false,
  "repairs_applied": false
}
{"case_key": [1, 30575], "review": {"source_task": 1, "annotation_id": 30575, "decision": "regenerate", "bbox_assessment": "too_loose", "polygon_assessment": "plausible", "observations": ["Trong crop chưa gắn nhãn, chai nằm chéo; điểm ngoài cùng bên trái là phần thân chai ở vùng thấp hơn giữa ảnh, cách xa mép trái của bbox đỏ.", "Cạnh trái bbox đỏ kéo dài qua một khoảng nền lớn và cả phần bao bì trắng/đỏ phía sau chai; không thấy mảnh chai nào gần cạnh đó.", "Các cạnh trên, phải và dưới của bbox nhìn chung sát với các đầu mút chai; phần kéo dư rõ rệt tập trung ở cạnh trái."], "reason": "BBox đỏ mở rộng quá xa về bên trái so với các pixel chai nhìn thấy; đây không chỉ là góc trống do chai bị xoay. Polygon cyan bao

## 8. Kiểm tra nội dung pilot và chạy tiếp nếu đạt

Đối chiếu cả 10 nhận xét với PNG/cutout tương ứng: đúng đối tượng/vị trí, không suy ra lỗi chỉ từ IoU, không nhầm khoảng trống của bbox xoay nghiêng thành lỗi, không tuyên bố đã có mask/đã sửa nhãn. Ghi audit theo đúng fingerprints trong `ai/gpt-6-luna_v1/pilot_quality_audit.json`: `passed`, `pilot_fingerprints`, `reviewer`, và `case_audits` chứa `fingerprint,grounded,notes` từng case. Audit cần đánh giá nội dung thực, không tự điền PASS từ schema hoặc confidence.

Cell dưới chỉ gọi những case ngoài pilot khi `AI_RUN_REMAINING=True` và audit đạt. Nếu chưa đạt, giữ output pilot và điều chỉnh prompt/đầu vào hoặc review thủ công trước; không tự gửi cả 222 ảnh. Confidence là model tự báo, không phải xác suất correctness đã hiệu chuẩn.

**Kết quả pilot ngày 07/10:** schema/API đạt, audit nội dung **FAIL** ở case `(4,15271)` và `(4,9534)`; 212 case còn lại chưa gửi. Nhận xét cuối 10 case gồm 6 đề xuất regenerate, 3 keep, 1 pending; chưa chấp nhận/sửa nhãn. [Báo cáo pilot](../../docs/data_preprocessing/reports/ai-review-pilot-2026-10-07.md).


In [3]:
AI_AUDIT_PATH=AI_OUT/'pilot_quality_audit.json'
ai_quality_audit=json.loads(AI_AUDIT_PATH.read_text()) if AI_AUDIT_PATH.exists() else {}
ai_pilot_quality_pass=bool(ai_pilot_results) and pilot_gate_passed(ai_pilot_results,ai_quality_audit)
print('Pilot visual-quality gate:', 'PASS' if ai_pilot_quality_pass else 'FAIL' if AI_AUDIT_PATH.exists() else 'PENDING')
if ai_quality_audit:print(json.dumps(ai_quality_audit,ensure_ascii=False,indent=2))
if AI_RUN_REMAINING:
    assert ai_pilot_quality_pass,'Continuation requires a visual audit of this exact pilot'
    pilot_keys={tuple(r['case_key']) for r in ai_pilot_results}
    remaining=[r for r in AI_CASES if (r['source_task'],r['annotation_id']) not in pilot_keys]
    assert len(remaining)==212
    with ThreadPoolExecutor(max_workers=AI_SETTINGS['workers']) as pool:
        ai_remaining_results=list(pool.map(ai_review_one,remaining))
    write_json(AI_ARTIFACT/'remaining_results.json',ai_remaining_results)
    print('Remaining cases:',len(ai_remaining_results),'| newly called:',sum(not r['cached'] for r in ai_remaining_results))
else:print('Remaining API calls disabled; no extra images sent.')


Pilot visual-quality gate: FAIL
{
  "passed": false,
  "reviewer": "Codex — independent visual inspection of all 10 cards, original cutouts and native crop",
  "evaluated_at": "2026-10-07T20:50:46+07:00",
  "prompt_version": "rpc-bbox-polygon-review-v3",
  "pilot_fingerprints": [
    "35527e7784d1b149f46763063f3d615c8ffa36e3ae14a11c2de9018ca0286d2f",
    "bdcb81d89e1967c90494959ba66ea9e5b7d07ed34b99dd150942542b56cc3223",
    "0c77a39b8a5d37c5be7348c1bf8500e68c85dbaad5fe68ca1365fc8ff91f59dc",
    "ac4ad9e85bbf4b92a3ad27e7e2ba323e369d357a2e2972ee2ad036af156c9fcb",
    "12e6b8ae369af8db740cde6a2ad7832ea297785d050422017f10d6daf988ea03",
    "35ac1e0df4250a20d4c8c26bc84965366fc00911c60cac03fa0f5a1afd8d7b7f",
    "ff4f69a4dfc2f67cdd0af0fb2c64dd51b355d08bb763a20d17059d2e53ddfcb8",
    "48c7154fa4cc00cc22218e4bb8b6a72d6941e3829dcdd5163cd1e09c0f5773bc",
    "45ace26c74dec6b88b737fb8e0b2b26868f8db4509413bbd44ef7accd6bba683",
    "70f4727bbf320c605ccc2857dacf7eab3fcf51dfd47ef9e665759f939408413c"


## 9. Nghiệm thu API workflow và kết thúc

Phân biệt các trạng thái: model đã trả nhận xét; pilot đã được kiểm tra nội dung; quyết định cuối của người review; nhãn đã được sửa. Workflow AI này chỉ sinh đề xuất review. CSV manual và source checksum phải giữ nguyên; receipt/usage/output/run metadata phải được lưu. Phần sửa/tái sinh và nghiệm thu mask/bbox/provenance vẫn là công việc tiếp theo.


In [4]:
ai_results=ai_pilot_results+ai_remaining_results
assert len({tuple(r['case_key']) for r in ai_results})==len(ai_results)
assert file_sha256(AI_MANUAL_CSV)==AI_MANUAL_HASH
for task,digest in ai_run['source_sha256'].items():assert file_sha256(AI_ROOT/f'synth/group/task{task}/annotations.json')==digest
if ai_remaining_results:assert len(ai_results)==222 and ai_pilot_quality_pass
ai_run.update({'finished_at':timestamp(),'status':'AI_SUGGESTIONS_COMPLETE_REPAIR_PENDING' if len(ai_results)==222 else
              'PILOT_QUALITY_FAILED_NO_CONTINUATION' if ai_pilot_results and AI_AUDIT_PATH.exists() and not ai_pilot_quality_pass else
              'PILOT_READY_FOR_CONTINUATION' if ai_pilot_quality_pass else 'PILOT_COMPLETE_QUALITY_PENDING' if ai_pilot_results else 'API_DISABLED'})
from datetime import datetime
ai_run['duration_seconds']=(datetime.fromisoformat(ai_run['finished_at'])-datetime.fromisoformat(ai_run['started_at'])).total_seconds()
ai_summary={'run':ai_run,'cases_completed':len(ai_results),'pilot_cases':len(ai_pilot_results),
            'pilot_quality_pass':ai_pilot_quality_pass,'new_api_requests':sum(not r['cached'] for r in ai_results),
            'cached_results':sum(r['cached'] for r in ai_results),
            'experiment_ledger':str((AI_OUT/'experiment_ledger.json').relative_to(AI_REPO)),
            'decision_counts':dict(Counter(r['review']['decision'] for r in ai_results)),
            'needs_human':sum(r['review']['needs_human'] for r in ai_results),
            'usage':{k:sum(r['usage'][k] for r in ai_results) for k in ['input_tokens','output_tokens','total_tokens']},
            'estimated_cost_usd':sum(r['estimated_cost_usd'] for r in ai_results),
            'estimated_new_cost_usd':sum(r['estimated_cost_usd'] for r in ai_results if not r['cached']),
            'manual_decision_csv_modified':False,'label_repairs_applied':False,
            'results':str((AI_ARTIFACT/'all_results.json').relative_to(AI_REPO))}
write_json(AI_ARTIFACT/'all_results.json',ai_results);write_json(AI_ARTIFACT/'summary.json',ai_summary)
write_json(AI_OUT/'latest_summary.json',ai_summary)
print(json.dumps(ai_summary,ensure_ascii=False,indent=2))


{
  "run": {
    "run_id": "10897daf-1644-4404-873f-d485ba24bda0",
    "purpose": "OpenAI pilot 10 then conditional remaining reviews",
    "started_at": "2026-10-07T21:01:13+07:00",
    "finished_at": "2026-10-07T21:01:14+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "git_dirty": true,
    "settings": {
      "model": "gpt-6-luna",
      "prompt_version": "rpc-bbox-polygon-review-v3",
      "endpoint": "https://api.openai.com/v1/responses",
      "reasoning_effort": "low",
      "image_detail": "high",
      "cutout_max_side": 512,
      "max_output_tokens": 3000,
      "pilot_cases": 10,
      "workers": 4,
      "require_visual_pilot_audit": true,
      "apply_label_repairs": false,
      "write_manual_decisions": false,
      "pricing_checked_on": "2026-10-07",
      "pricing_mode": "standard_short_context",
      "usd_per_million_tokens": {
        "input": 0.1,
        "cache

## 10. Kiểm thử nội dung của 10 nhận xét pilot

Đối chiếu phản hồi đã lưu với ảnh native, cutout và chênh lệch các cạnh bbox/polygon. Không gọi thêm API. Tách ba câu hỏi: mô tả có được pixels/metadata hỗ trợ không; đề xuất có đủ căn cứ để dùng sàng lọc không; mức chắc chắn có phù hợp với bằng chứng không. Thiếu owner-mask gốc nên không tính accuracy nhãn hoặc tự chứng nhận mask đúng.

Cell chuẩn bị lưu input hashes, crop native/edge zoom và thời điểm bắt đầu. Sau kiểm tra trực quan, bộ đối chiếu sẽ khóa theo case, source hash và nội dung nhận xét đã chấm; thay model/prompt/output cần đối chiếu mới. Cảnh báo geometry là chênh giữa hai biểu diễn, không mặc định là bbox sai.


In [1]:
import json,uuid,hashlib,sys,platform,os
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from PIL import Image
CT_REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
if str(CT_REPO) not in sys.path:sys.path.insert(0,str(CT_REPO))
from tools.ai_annotation_review import write_json,file_sha256,parse_review
from tools.annotation_review import review_crop_box
CT_NOW=lambda:datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds')
CT_STARTED_AT=CT_NOW()
CT_REVIEW_ROOT=CT_REPO/'data/processed/synthetic_annotation_review_v1'
CT_AI_ROOT=CT_REVIEW_ROOT/'ai/gpt-6-luna_v1'
CT_SUMMARY=json.loads((CT_AI_ROOT/'latest_summary.json').read_text())
CT_RESULTS_PATH=CT_REPO/CT_SUMMARY['results'];CT_RESULTS=json.loads(CT_RESULTS_PATH.read_text())
CT_RESULTS_HASH=file_sha256(CT_RESULTS_PATH)
assert len(CT_RESULTS)==10 and {r['prompt_version'] for r in CT_RESULTS}=={'rpc-bbox-polygon-review-v3'}
CT_QUEUE={(r['source_task'],r['annotation_id']):r for r in json.loads((CT_REVIEW_ROOT/'review_queue.json').read_text())}
CT_RAW=Path(os.getenv('AUTOCHECKOUT_DATA_ROOT',str(CT_REPO/'data/archive'))).resolve()
CT_RUN_ID=str(uuid.uuid4());CT_ARTIFACT=CT_REPO/'runs/data_preprocessing/content_review'/CT_RUN_ID
CT_ARTIFACT.mkdir(parents=True,exist_ok=True)
CT_RUN={'run_id':CT_RUN_ID,'purpose':'content verification against native pixels and geometry',
        'started_at':CT_STARTED_AT,'finished_at':None,'timezone':'Asia/Bangkok','python':sys.version.split()[0],
        'host':platform.node(),'api_calls':0,'owner_masks_available':False,'status':'PREPARED'}
CT_MANUAL_HASH=file_sha256(CT_REVIEW_ROOT/'review_decisions.csv')
CT_SOURCE_HASHES={task:file_sha256(CT_RAW/f'synth/group/task{task}/annotations.json') for task in range(1,5)}
CT_ASSET_HASHES={}
CT_GEOMETRY=[]
for receipt in CT_RESULTS:
    key=tuple(receipt['case_key']);q=CT_QUEUE[key]
    assert q['source_sha256']==receipt['source_sha256']==CT_SOURCE_HASHES[key[0]]
    assert parse_review(receipt['response'],q)==receipt['review']
    bbox=q['bbox'];poly=q['polygon_bbox'];x,y,w,h=bbox;px,py,pw,ph=poly
    gaps=[px-x,py-y,x+w-px-pw,y+h-py-ph]
    edge_index=max(range(4),key=lambda i:abs(gaps[i]));edge=['left','top','right','bottom'][edge_index]
    photo=CT_RAW/q['image_file'];cutout=CT_RAW/q['cutout']
    for asset in [photo,cutout]:CT_ASSET_HASHES[str(asset.relative_to(CT_RAW))]=file_sha256(asset)
    native=CT_ARTIFACT/f'task{key[0]}_ann{key[1]}_native.png'
    edge_path=CT_ARTIFACT/f'task{key[0]}_ann{key[1]}_{edge}_zoom.png'
    with Image.open(photo) as im:
        rectangle_polygon=[[px,py,px+pw-1,py,px+pw-1,py+ph-1,px,py+ph-1]]
        crop_box=review_crop_box(bbox,rectangle_polygon,im.size)
        im.convert('RGB').crop(crop_box).save(native)
        if edge=='left':roi=(x-16,y-16,px+16,y+h+16)
        elif edge=='top':roi=(x-16,y-16,x+w+16,py+16)
        elif edge=='right':roi=(px+pw-16,y-16,x+w+16,y+h+16)
        else:roi=(x-16,py+ph-16,x+w+16,y+h+16)
        roi=(max(0,int(roi[0])),max(0,int(roi[1])),min(im.width,int(roi[2])),min(im.height,int(roi[3])))
        small=im.convert('RGB').crop(roi);small.resize((small.width*3,small.height*3),Image.Resampling.NEAREST).save(edge_path)
    CT_GEOMETRY.append({'case_key':list(key),'bbox':bbox,'polygon_bbox':poly,'iou':q['iou'],
        'edge_gaps_ltrb_px':gaps,'largest_gap_edge':edge,'largest_gap_px':max(abs(v) for v in gaps),
        'largest_gap_fraction':max(abs(gaps[0])/w,abs(gaps[1])/h,abs(gaps[2])/w,abs(gaps[3])/h),
        'image_file':q['image_file'],'cutout':q['cutout'],'native':str(native.relative_to(CT_REPO)),
        'edge_zoom':str(edge_path.relative_to(CT_REPO)),'source_sha256':q['source_sha256']})
write_json(CT_ARTIFACT/'geometry.json',CT_GEOMETRY);write_json(CT_ARTIFACT/'source_asset_sha256.json',CT_ASSET_HASHES)
write_json(CT_ARTIFACT/'run.json',CT_RUN)
print(json.dumps({'run':CT_RUN,'cases':CT_GEOMETRY},ensure_ascii=False,indent=2))


{
  "run": {
    "run_id": "3e68a2cc-2e2e-4ba8-adcd-4189c5d739e4",
    "purpose": "content verification against native pixels and geometry",
    "started_at": "2026-10-07T22:15:30+07:00",
    "finished_at": null,
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "api_calls": 0,
    "owner_masks_available": false,
    "status": "PREPARED"
  },
  "cases": [
    {
      "case_key": [
        1,
        30575
      ],
      "bbox": [
        865,
        535,
        318,
        299
      ],
      "polygon_bbox": [
        907.0,
        535.0,
        276.0,
        295.0
      ],
      "iou": 0.8563134978229228,
      "edge_gaps_ltrb_px": [
        42.0,
        0.0,
        0.0,
        4.0
      ],
      "largest_gap_edge": "left",
      "largest_gap_px": 42.0,
      "largest_gap_fraction": 0.1320754716981132,
      "image_file": "synth/group/task1/images/002537.jpg",
      "cutout": "cutouts/sam/objects/073_drink/6928804010114_camera1-10.png",
      "n

### Đối chiếu nội dung và kết thúc

Bộ đối chiếu [annotation_review_content_reference_v1.json](../../configs/data/annotation_review_content_reference_v1.json) được lập sau khi xem 10 crop native, cutout, geometry và edge zoom. Hash nhận xét khóa đúng output đã chấm; khi output/prompt đổi, cần chấm lại, không tự dùng điểm cũ cho model mới.

`SUPPORTED_TRIAGE` nghĩa là đề xuất có thể dùng chọn công việc kiểm tra tiếp, không phải nhãn đã đúng. `PARTIAL` là mô tả hướng lệch có cơ sở nhưng bỏ qua bằng chứng/giới hạn quan trọng. `UNSUPPORTED_CERTAINTY` đánh dấu kết luận mạnh hơn bằng chứng và cần review thêm. Tỉ lệ tương thích với rubric không phải accuracy segmentation/bbox.


In [2]:
from collections import Counter
CT_REFERENCE_PATH=CT_REPO/'configs/data/annotation_review_content_reference_v1.json'
CT_REFERENCE=json.loads(CT_REFERENCE_PATH.read_text())
refs={tuple(r['case_key']):r for r in CT_REFERENCE['cases']}
assert set(refs)=={tuple(r['case_key']) for r in CT_RESULTS}
geometry_by_key={tuple(r['case_key']):r for r in CT_GEOMETRY}
CT_ROWS=[]
for receipt in CT_RESULTS:
    key=tuple(receipt['case_key']);ref=refs[key];review=receipt['review'];g=geometry_by_key[key]
    digest=hashlib.sha256(json.dumps(review,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
    assert ref['source_sha256']==receipt['source_sha256'] and ref['fingerprint']==receipt['fingerprint']
    assert ref['review_sha256']==digest,'Output changed: perform a new content review, do not reuse the old grade'
    CT_ROWS.append({'case_key':list(key),'decision':review['decision'],'confidence_self_reported':review['confidence'],
        'model_needs_human':review['needs_human'],'observation_verdict':ref['observation_verdict'],
        'decision_verdict':ref['decision_verdict'],'allowed_triage_decisions':ref['allowed_triage_decisions'],
        'triage_decision_agrees':review['decision'] in ref['allowed_triage_decisions'],
        'needs_human_expected':ref['needs_human_expected'],'notes':ref['notes'],
        'edge_gaps_ltrb_px':g['edge_gaps_ltrb_px'],'native':g['native'],'edge_zoom':g['edge_zoom']})
# Compare decisions across historical prompts descriptively; these are different rubrics/settings.
CT_ROUNDS={}
for path in (CT_AI_ROOT/'cases').glob('*.json'):
    receipt=json.loads(path.read_text());version=receipt['prompt_version'];key=tuple(receipt['case_key'])
    try:decision=parse_review(receipt['response'],CT_QUEUE[key])['decision']
    except ValueError:decision='INVALID_OR_INCOMPLETE'
    CT_ROUNDS.setdefault(version,{})[key]=decision
CT_TRANSITIONS=[{'case_key':r['case_key'],'decisions':{v:rows.get(tuple(r['case_key'])) for v,rows in sorted(CT_ROUNDS.items())}} for r in CT_ROWS]
CT_CHANGED_V1_V3=sum(r['decisions'].get('rpc-bbox-polygon-review-v1')!=r['decisions'].get('rpc-bbox-polygon-review-v3') for r in CT_TRANSITIONS)
CT_OBSERVATION_COUNTS=dict(Counter(r['observation_verdict'] for r in CT_ROWS))
CT_DECISION_COUNTS=dict(Counter(r['decision_verdict'] for r in CT_ROWS))
CT_AGREED=sum(r['triage_decision_agrees'] for r in CT_ROWS)
CT_MISSED_HUMAN=[r['case_key'] for r in CT_ROWS if r['needs_human_expected'] and not r['model_needs_human']]
CT_PASS=all(r['observation_verdict']=='SUPPORTED' and r['triage_decision_agrees'] for r in CT_ROWS)
assert file_sha256(CT_RESULTS_PATH)==CT_RESULTS_HASH
assert file_sha256(CT_REVIEW_ROOT/'review_decisions.csv')==CT_MANUAL_HASH
assert {task:file_sha256(CT_RAW/f'synth/group/task{task}/annotations.json') for task in CT_SOURCE_HASHES}==CT_SOURCE_HASHES
assert {name:file_sha256(CT_RAW/name) for name in CT_ASSET_HASHES}==CT_ASSET_HASHES
CT_RUN.update({'finished_at':CT_NOW(),'status':'CONTENT_PASS' if CT_PASS else 'CONTENT_FAIL'})
CT_RUN['duration_seconds']=(datetime.fromisoformat(CT_RUN['finished_at'])-datetime.fromisoformat(CT_RUN['started_at'])).total_seconds()
CT_REPORT={'run':CT_RUN,'cases':len(CT_ROWS),'observation_counts':CT_OBSERVATION_COUNTS,'decision_counts':CT_DECISION_COUNTS,
    'triage_decision_agreement_count':CT_AGREED,'triage_decision_agreement_fraction':CT_AGREED/len(CT_ROWS),
    'missed_human_review_cases':CT_MISSED_HUMAN,'content_pass':CT_PASS,'ground_truth_accuracy':None,
    'label_repairs_accepted':0,'new_api_calls':0,'v1_to_v3_decision_changes':CT_CHANGED_V1_V3,
    'cross_prompt_comparison_is_not_a_same_prompt_repeatability_test':True,
    'reference':str(CT_REFERENCE_PATH.relative_to(CT_REPO)),'reference_sha256':file_sha256(CT_REFERENCE_PATH),
    'input_results':str(CT_RESULTS_PATH.relative_to(CT_REPO)),'input_results_sha256':file_sha256(CT_RESULTS_PATH),
    'notes':'Scores describe this reviewer rubric on these 10 cached outputs; owner masks absent, no label accuracy claimed.'}
write_json(CT_ARTIFACT/'case_content_checks.json',CT_ROWS);write_json(CT_ARTIFACT/'prompt_decision_comparison.json',CT_TRANSITIONS)
write_json(CT_ARTIFACT/'summary.json',CT_REPORT);write_json(CT_ARTIFACT/'run.json',CT_RUN)
write_json(CT_AI_ROOT/'content_test_latest.json',CT_REPORT)
import html
cards=[]
for r in CT_ROWS:
    key=tuple(r['case_key']);model_review=next(x['review'] for x in CT_RESULTS if tuple(x['case_key'])==key)
    label=f"Task {key[0]} / ann {key[1]} — {r['decision_verdict']}"
    native_name=Path(r['native']).name;edge_name=Path(r['edge_zoom']).name
    obs=''.join('<li>'+html.escape(x)+'</li>' for x in model_review['observations'])
    cards.append('<details open><summary>'+html.escape(label)+'</summary>'
        '<p><b>Model:</b> '+html.escape(r['decision'])+'; confidence tự báo '+str(r['confidence_self_reported'])+'</p>'
        '<p><b>Đối chiếu:</b> '+html.escape(r['notes'])+'</p>'
        '<p>Chênh cạnh L/T/R/B (px): '+html.escape(str(r['edge_gaps_ltrb_px']))+'</p>'
        '<div class="photos"><a href="'+html.escape(native_name)+'"><img src="'+html.escape(native_name)+'" alt="Crop native"></a>'
        '<a href="'+html.escape(edge_name)+'"><img src="'+html.escape(edge_name)+'" alt="Zoom cạnh lệch"></a></div>'
        '<p><b>Nhận xét gốc:</b> '+html.escape(model_review['reason'])+'</p><ul>'+obs+'</ul></details>')
(CT_ARTIFACT/'content_report.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Content test RPC</title>'
    '<style>body{font:16px system-ui;max-width:1200px;margin:24px auto;padding:0 16px;color:#222}'
    'details{border:1px solid #ccc;padding:14px;margin:16px 0}summary{font-weight:600;cursor:pointer}'
    '.photos{display:flex;gap:20px;align-items:flex-start}.photos a{width:50%}.photos img{max-width:100%;max-height:650px}'
    '@media(max-width:700px){.photos{display:block}.photos a{width:100%}}</style>'
    '<h1>Đối chiếu nội dung 10 case</h1><p>7 đề xuất phù hợp để sàng lọc; 3 kết luận mạnh hơn bằng chứng. '
    'Thiếu owner-mask: đây không phải accuracy nhãn. Không gọi thêm API hoặc sửa nhãn.</p>'+''.join(cards)+'</html>')
CT_REPORT['html_report']=str((CT_ARTIFACT/'content_report.html').relative_to(CT_REPO))
write_json(CT_ARTIFACT/'summary.json',CT_REPORT);write_json(CT_AI_ROOT/'content_test_latest.json',CT_REPORT)
print(json.dumps(CT_REPORT,ensure_ascii=False,indent=2))
print('Case | model decision | observation | decision content | edge gap L/T/R/B | content notes')
for r in CT_ROWS:
    print(f"{r['case_key']} | {r['decision']} | {r['observation_verdict']} | {r['decision_verdict']} | {r['edge_gaps_ltrb_px']} | {r['notes']}")
print('Report artifacts:',str(CT_ARTIFACT.relative_to(CT_REPO)))


{
  "run": {
    "run_id": "3e68a2cc-2e2e-4ba8-adcd-4189c5d739e4",
    "purpose": "content verification against native pixels and geometry",
    "started_at": "2026-10-07T22:15:30+07:00",
    "finished_at": "2026-10-07T22:15:31+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "api_calls": 0,
    "owner_masks_available": false,
    "status": "CONTENT_FAIL",
    "duration_seconds": 1.0
  },
  "cases": 10,
  "observation_counts": {
    "SUPPORTED": 7,
    "PARTIAL": 3
  },
  "decision_counts": {
    "SUPPORTED_TRIAGE": 7,
    "UNSUPPORTED_CERTAINTY": 3
  },
  "triage_decision_agreement_count": 7,
  "triage_decision_agreement_fraction": 0.7,
  "missed_human_review_cases": [
    [
      2,
      7318
    ],
    [
      4,
      15271
    ],
    [
      4,
      9534
    ]
  ],
  "content_pass": false,
  "ground_truth_accuracy": null,
  "label_repairs_accepted": 0,
  "new_api_calls": 0,
  "v1_to_v3_decision_changes": 7,
  "cross_prompt_comparison_is_n

## 11. Pilot Sol trên 10 ảnh mới — so sánh chất lượng sàng lọc

Theo yêu cầu ngày 07/10/2026, dùng đúng **gpt-6.1-sol** trên 10 ảnh khác pilot Luna. Giữ prompt v3, reasoning `low`, hai ảnh/case (card đối chiếu và cutout), detail `high`, tối đa 3000 output tokens. Loại cả case key lẫn đường dẫn ảnh đã gửi Luna; lựa chọn cố định có đủ 4 task nguồn và mức lệch đa dạng. Các task nguồn synth này khác protocol train 5 task đã chốt.

Chạy độc lập ba cell phần 11–12, không cần PyTorch/CUDA. Đặt `SOL_RUN_PILOT=True` hoặc biến môi trường `AUTOCHECKOUT_REVIEW_SOL_PILOT=1` để gọi API. Mặc định tắt; sau khi đã có receipts, chạy lại dùng cache mà không cần key. Kết quả và chi phí Sol lưu riêng, không thay lịch sử Luna. Có checksum nguồn, snapshot config/prompt, run ID và thời điểm bắt đầu/kết thúc.

Bộ [đối chiếu Sol](../../configs/data/annotation_review_sol_content_reference_v1.json) ghi kỳ vọng trực quan trước khi gửi API, rồi khóa hash output khi kiểm thử. Nhãn CSV vẫn cần người review. So sánh Luna/Sol ở đây dùng **hai tập ảnh khác nhau**; điểm chỉ phản ánh mức phù hợp rubric trên từng pilot, chưa chứng minh model nào hơn trên cùng dữ liệu. [OpenAI Docs](https://developers.openai.com/api/docs/models/gpt-6.1-sol); [cấu hình Sol](../../configs/data/annotation_review_sol_pilot_v1.json).


In [1]:
import os, sys, json, uuid, platform, subprocess, hashlib, html
from pathlib import Path
from datetime import datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
SOL_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pdp/main.py').exists())
if str(SOL_REPO) not in sys.path:
    sys.path.insert(0, str(SOL_REPO))
from tools.ai_annotation_review import (PROMPT_VERSION, PROMPT, select_pilot, load_api_key,
    create_response, review_case, parse_review, pilot_gate_passed, file_sha256, write_json, timestamp)
SOL_RUN_PILOT = os.getenv('AUTOCHECKOUT_REVIEW_SOL_PILOT', '0') == '1'  # Có thể đổi thành True.
SOL_RAW = Path(os.getenv('AUTOCHECKOUT_DATA_ROOT', str(SOL_REPO / 'data/archive'))).resolve()
SOL_REVIEW_ROOT = SOL_REPO / 'data/processed/synthetic_annotation_review_v1'
SOL_OUT = SOL_REVIEW_ROOT / 'ai/gpt-6.1-sol_pilot_v1'
SOL_OUT.mkdir(parents=True, exist_ok=True)
SOL_SETTINGS = json.loads((SOL_REPO / 'configs/data/annotation_review_sol_pilot_v1.json').read_text())
assert SOL_SETTINGS['model'] == 'gpt-6.1-sol' and SOL_SETTINGS['prompt_version'] == PROMPT_VERSION
assert SOL_SETTINGS['reasoning_effort'] == 'low' and SOL_SETTINGS['max_output_tokens'] == 3000
SOL_VISUAL = json.loads((SOL_REVIEW_ROOT / 'review_summary.json').read_text())
assert SOL_VISUAL['visual_acceptance'] == 'PASS'
SOL_QUEUE = {(r['source_task'], r['annotation_id']): r
             for r in json.loads((SOL_REVIEW_ROOT / 'review_queue.json').read_text())}
SOL_ALL = json.loads((SOL_REPO / SOL_VISUAL['visual_manifest']).read_text())
assert len(SOL_ALL) == len(SOL_QUEUE) == 222
for r in SOL_ALL:
    q = SOL_QUEUE[r['source_task'], r['annotation_id']]
    assert q['source_sha256'] == r['source_sha256']
    r.update({k: q[k] for k in ['bbox', 'polygon_bbox']})
SOL_OLD_RESULTS_PATH = SOL_REVIEW_ROOT / 'ai/gpt-6-luna_v1/pilot_results.json'
SOL_OLD_RESULTS = json.loads(SOL_OLD_RESULTS_PATH.read_text())
SOL_OLD_KEYS = {tuple(r['case_key']) for r in SOL_OLD_RESULTS}
SOL_OLD_IMAGES = {SOL_QUEUE[k]['image_file'] for k in SOL_OLD_KEYS}
SOL_ELIGIBLE = [r for r in SOL_ALL if (r['source_task'], r['annotation_id']) not in SOL_OLD_KEYS
                and r['image_file'] not in SOL_OLD_IMAGES]
SOL_PILOT = select_pilot(SOL_ELIGIBLE, count=SOL_SETTINGS['pilot_cases'])
assert len(SOL_PILOT) == len({r['image_file'] for r in SOL_PILOT}) == 10
assert not ({r['image_file'] for r in SOL_PILOT} & SOL_OLD_IMAGES)
assert {r['source_task'] for r in SOL_PILOT} == {1, 2, 3, 4}
SOL_SELECTION_PATH = SOL_OUT / 'pilot_selection.json'
if SOL_SELECTION_PATH.exists():
    assert json.loads(SOL_SELECTION_PATH.read_text()) == SOL_PILOT, 'Pilot selection changed; create a new experiment version.'
else:
    write_json(SOL_SELECTION_PATH, SOL_PILOT)
SOL_ARTIFACT = SOL_REPO / 'runs/data_preprocessing/sol_review' / str(uuid.uuid4())
SOL_ARTIFACT.mkdir(parents=True, exist_ok=True)
SOL_MANUAL_HASH = file_sha256(SOL_REVIEW_ROOT / 'review_decisions.csv')
SOL_SOURCE_HASHES = {t: file_sha256(SOL_RAW / f'synth/group/task{t}/annotations.json') for t in range(1, 5)}
assert {str(t): h for t, h in SOL_SOURCE_HASHES.items()} == SOL_VISUAL['source_sha256']
SOL_VISUAL_ASSETS = json.loads((SOL_REPO / Path(SOL_VISUAL['visual_manifest']).parent / 'source_asset_sha256.json').read_text())
SOL_ASSET_HASHES = {}
SOL_GEOMETRY = []
sol_run = {'run_id': SOL_ARTIFACT.name, 'purpose': 'Sol pilot on 10 photos excluded from Luna, then content verification',
    'started_at': timestamp(), 'finished_at': None, 'timezone': 'Asia/Bangkok', 'python': sys.version.split()[0],
    'host': platform.node(), 'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()),
    'settings': SOL_SETTINGS, 'api_enabled': SOL_RUN_PILOT, 'cuda_executed': False,
    'code_sha256': file_sha256(SOL_REPO / 'tools/ai_annotation_review.py'),
    'excluded_luna_results_sha256': file_sha256(SOL_OLD_RESULTS_PATH), 'status': 'PREPARED'}
for case in SOL_PILOT:
    key = (case['source_task'], case['annotation_id'])
    assert file_sha256(SOL_REPO / case['preview']) == case['preview_sha256']
    for name in [case['image_file'], case['cutout']]:
        SOL_ASSET_HASHES[name] = file_sha256(SOL_RAW / name)
        assert SOL_ASSET_HASHES[name] == SOL_VISUAL_ASSETS[name]
    x, y, w, h = case['bbox']; px, py, pw, ph = case['polygon_bbox']
    gaps = [px-x, py-y, x+w-px-pw, y+h-py-ph]
    side = ['left', 'top', 'right', 'bottom'][max(range(4), key=lambda i: abs(gaps[i]))]
    stem = f'task{key[0]}_ann{key[1]}'
    with Image.open(SOL_RAW / case['image_file']) as im:
        im.convert('RGB').crop(case['crop_box']).save(SOL_ARTIFACT / (stem + '_native.png'))
        if side == 'left': roi = (x-16, y-16, px+16, y+h+16)
        elif side == 'top': roi = (x-16, y-16, x+w+16, py+16)
        elif side == 'right': roi = (px+pw-16, y-16, x+w+16, y+h+16)
        else: roi = (x-16, py+ph-16, x+w+16, y+h+16)
        roi = (max(0, int(roi[0])), max(0, int(roi[1])), min(im.width, int(roi[2])), min(im.height, int(roi[3])))
        edge = im.convert('RGB').crop(roi)
        edge.resize((edge.width*3, edge.height*3), Image.Resampling.NEAREST).save(SOL_ARTIFACT / (stem + '_edge.png'))
    with Image.open(SOL_RAW / case['cutout']) as im:
        rgba = im.convert('RGBA'); rgba.thumbnail((512, 512), Image.Resampling.LANCZOS)
        white = Image.new('RGB', rgba.size, 'white'); white.paste(rgba, mask=rgba.getchannel('A'))
        white.save(SOL_ARTIFACT / (stem + '_cutout.png'))
    SOL_GEOMETRY.append({'case_key': list(key), 'iou': case['iou'], 'bbox': case['bbox'],
        'polygon_bbox': case['polygon_bbox'], 'edge_gaps_ltrb_px': gaps, 'largest_gap_edge': side,
        'native': stem + '_native.png', 'edge_zoom': stem + '_edge.png', 'cutout': stem + '_cutout.png',
        'preview': case['preview'], 'image_file': case['image_file']})
write_json(SOL_ARTIFACT / 'run.json', sol_run)
write_json(SOL_ARTIFACT / 'pilot_selection.json', SOL_PILOT)
write_json(SOL_ARTIFACT / 'geometry.json', SOL_GEOMETRY)
write_json(SOL_ARTIFACT / 'source_asset_sha256.json', SOL_ASSET_HASHES)
(SOL_ARTIFACT / 'prompt.txt').write_text(PROMPT)
print(json.dumps({'run': sol_run, 'selected_cases': [r['case_key'] for r in SOL_GEOMETRY],
                  'overlap_with_luna_photos': 0}, ensure_ascii=False, indent=2))


{
  "run": {
    "run_id": "11cf4169-bd69-4331-9726-768c48afd9ef",
    "purpose": "Sol pilot on 10 photos excluded from Luna, then content verification",
    "started_at": "2026-10-07T23:11:57+07:00",
    "finished_at": null,
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "git_dirty": true,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-bbox-polygon-review-v3",
      "endpoint": "https://api.openai.com/v1/responses",
      "reasoning_effort": "low",
      "image_detail": "high",
      "cutout_max_side": 512,
      "max_output_tokens": 3000,
      "pilot_cases": 10,
      "workers": 2,
      "exclude_previous_model": "gpt-6-luna",
      "require_visual_pilot_audit": true,
      "apply_label_repairs": false,
      "write_manual_decisions": false,
      "pricing_checked_on": "2026-10-07",
      "pricing_mode": "standard_short_context",
      "usd_per_million_token

In [2]:
sol_results = []
sol_errors = []
sol_have_receipts = all(list((SOL_OUT / 'cases').glob(f"task{r['source_task']}_ann{r['annotation_id']}_*.json")) for r in SOL_PILOT)
if SOL_RUN_PILOT or sol_have_receipts:
    if SOL_RUN_PILOT:
        import urllib.request, urllib.error
        sol_key = load_api_key(SOL_REPO / '.env')
        request = urllib.request.Request('https://api.openai.com/v1/models/gpt-6.1-sol',
                                        headers={'Authorization': 'Bearer ' + sol_key})
        try:
            with urllib.request.urlopen(request, timeout=30) as response:
                assert json.load(response)['id'] == SOL_SETTINGS['model']
        except urllib.error.HTTPError as error:
            raise RuntimeError(f'Exact Sol model access check failed: HTTP {error.code}; credentials/error body omitted') from None
        except urllib.error.URLError:
            raise RuntimeError('Exact Sol model access check failed: network unavailable') from None
        sol_send = lambda payload: create_response(payload, sol_key)
    else:
        def sol_send(payload):
            raise RuntimeError('API disabled: matching receipt missing; set SOL_RUN_PILOT=True explicitly.')
    def sol_review_one(case):
        try:
            receipt = review_case(case, repo=SOL_REPO, raw_root=SOL_RAW, out_dir=SOL_OUT,
                                  send=sol_send, model=SOL_SETTINGS['model'])
            return receipt, None
        except (ValueError, RuntimeError) as error:
            return None, {'case_key': [case['source_task'], case['annotation_id']], 'error': str(error)}
    with ThreadPoolExecutor(max_workers=SOL_SETTINGS['workers']) as pool:
        for receipt, error in pool.map(sol_review_one, SOL_PILOT):
            if error:
                sol_errors.append(error); print('Not accepted:', error)
            else:
                sol_results.append(receipt)
                print(receipt['case_key'], receipt['review']['decision'], 'cache=', receipt['cached'])
    write_json(SOL_ARTIFACT / 'pilot_results.json', sol_results)
    write_json(SOL_OUT / 'pilot_results.json', sol_results)
else:
    print('Sol API disabled; no receipts yet. Set SOL_RUN_PILOT=True to run the authorized 10-image pilot.')
sol_run['new_requests'] = sum(not r['cached'] for r in sol_results)
sol_run['cached_results'] = sum(r['cached'] for r in sol_results)
sol_run['valid_results'] = len(sol_results)
sol_run['errors'] = sol_errors
sol_run['status'] = 'PILOT_READY_CONTENT_PENDING' if len(sol_results) == 10 else ('PILOT_FAILED' if sol_errors else 'DISABLED')
write_json(SOL_ARTIFACT / 'run.json', sol_run)
print('Valid results:', len(sol_results), '; proposed decisions:', dict(Counter(r['review']['decision'] for r in sol_results)))


[1, 37955] regenerate cache= True
[1, 16503] keep_with_justification cache= True
[2, 36910] regenerate cache= True
[2, 46063] keep_with_justification cache= True
[3, 26454] regenerate cache= True
[3, 15851] keep_with_justification cache= True
[4, 49601] regenerate cache= True
[4, 168] keep_with_justification cache= True
[3, 22011] regenerate cache= True
[2, 9162] pending cache= True
Valid results: 10 ; proposed decisions: {'regenerate': 5, 'keep_with_justification': 4, 'pending': 1}


## 12. Nghiệm thu nội dung Sol và đối chiếu với pilot Luna

Cell này không gọi API. Reference lưu từng kỳ vọng trước API, kết quả kiểm tra mô tả sau API và hash output. Chấm đề xuất sàng lọc, mô tả quan sát và việc xử lý bất định; thiếu owner-mask nên không chấm accuracy nhãn và không chấp nhận sửa nhãn. Gate chỉ PASS khi đủ 10 case và mọi nội dung đều đã đối chiếu phù hợp.

HTML của mỗi run chứa crop native, cutout, zoom cạnh lệch, card có nhãn, nhận xét Sol và ghi chú kiểm tra. Khi reference chưa hoàn tất hoặc output thay đổi, status giữ CONTENT_PENDING; không tự dùng điểm cũ cho phản hồi mới. Pilot khác nhau chỉ cho phép so sánh mô tả kết quả; muốn kết luận hơn/kém phải chạy cùng ảnh và cùng rubric.


In [3]:
SOL_REF_PATH = SOL_REPO / 'configs/data/annotation_review_sol_content_reference_v1.json'
SOL_REFERENCE = json.loads(SOL_REF_PATH.read_text())
sol_reference_by_key = {tuple(r['case_key']): r for r in SOL_REFERENCE['cases']}
assert set(sol_reference_by_key) == {(r['source_task'], r['annotation_id']) for r in SOL_PILOT}
sol_checks = []
sol_content_ready = len(sol_results) == 10 and SOL_REFERENCE['review_complete'] is True
for receipt in sol_results:
    key = tuple(receipt['case_key']); ref = sol_reference_by_key[key]; q = SOL_QUEUE[key]
    assert ref['source_sha256'] == receipt['source_sha256'] == q['source_sha256']
    assert ref['preview_sha256'] == receipt['preview_sha256']
    assert ref['image_sha256'] == SOL_ASSET_HASHES[q['image_file']]
    assert ref['cutout_sha256'] == receipt['cutout_sha256'] == SOL_ASSET_HASHES[q['cutout']]
    review = parse_review(receipt['response'], q)
    digest = hashlib.sha256(json.dumps(review, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if sol_content_ready:
        assert ref['fingerprint'] == receipt['fingerprint'] and ref['review_sha256'] == digest, 'Output changed: recheck content.'
        assert ref['observation_verdict'] in ['SUPPORTED', 'PARTIAL', 'UNSUPPORTED']
        assert len(ref['content_notes'].strip()) >= 30
        decision_ok = review['decision'] in ref['allowed_triage_decisions']
        uncertainty_ok = not ref['requires_uncertainty'] or (review['decision'] == 'pending' and review['needs_human'])
        grounded = ref['observation_verdict'] == 'SUPPORTED' and decision_ok and uncertainty_ok
    else:
        decision_ok = uncertainty_ok = grounded = None
    sol_checks.append({'case_key': list(key), 'fingerprint': receipt['fingerprint'], 'review_sha256': digest,
        'decision': review['decision'], 'needs_human': review['needs_human'],
        'observation_verdict': ref['observation_verdict'], 'decision_matches_reference': decision_ok,
        'uncertainty_handled': uncertainty_ok, 'grounded': grounded,
        'notes': ref['content_notes'] or 'Content inspection not completed.'})
sol_audit = {'passed': sol_content_ready and all(r['grounded'] for r in sol_checks),
    'reviewer': SOL_REFERENCE['reviewer'], 'reviewed_at': timestamp(),
    'pilot_fingerprints': [r['fingerprint'] for r in sol_results], 'case_audits': sol_checks,
    'owner_masks_available': False, 'ground_truth_accuracy': None, 'label_repairs_accepted': 0}
sol_gate = pilot_gate_passed(sol_results, sol_audit)
if sol_content_ready:
    write_json(SOL_OUT / 'pilot_quality_audit.json', sol_audit)
sol_luna_content = json.loads((SOL_REVIEW_ROOT / 'ai/gpt-6-luna_v1/content_test_latest.json').read_text())
sol_comparison = {'same_images': False, 'same_prompt_version': True, 'same_reasoning_effort': True,
    'luna_content_report': sol_luna_content['html_report'],
    'luna_cases': 10, 'luna_content_compatible_cases': sol_luna_content['triage_decision_agreement_count'],
    'sol_triage_decision_agreement_count': sum(r['decision_matches_reference'] is True for r in sol_checks) if sol_content_ready else None,
    'sol_cases': len(sol_results), 'sol_content_compatible_cases': sum(r['grounded'] is True for r in sol_checks) if sol_content_ready else None,
    'controlled_model_quality_comparison': False,
    'limitation': 'Different 10-image cohorts; these scores do not prove model superiority or label accuracy.'}
for task, digest in SOL_SOURCE_HASHES.items():
    assert file_sha256(SOL_RAW / f'synth/group/task{task}/annotations.json') == digest
for name, digest in SOL_ASSET_HASHES.items():
    assert file_sha256(SOL_RAW / name) == digest
assert file_sha256(SOL_REVIEW_ROOT / 'review_decisions.csv') == SOL_MANUAL_HASH
sol_run['finished_at'] = timestamp()
sol_run['duration_seconds'] = (datetime.fromisoformat(sol_run['finished_at']) - datetime.fromisoformat(sol_run['started_at'])).total_seconds()
sol_run['status'] = ('CONTENT_PASS' if sol_gate else 'CONTENT_FAIL') if sol_content_ready else sol_run['status']
sol_summary = {'run': sol_run, 'model': SOL_SETTINGS['model'],
    'results': str((SOL_ARTIFACT / 'pilot_results.json').relative_to(SOL_REPO)),
    'decision_counts': dict(Counter(r['review']['decision'] for r in sol_results)),
    'usage': {'input_tokens': sum(r['usage']['input_tokens'] for r in sol_results),
              'output_tokens': sum(r['usage']['output_tokens'] for r in sol_results),
              'reasoning_tokens': sum((r['usage'].get('output_tokens_details') or {}).get('reasoning_tokens', 0) for r in sol_results)},
    'estimated_cost_usd': sum(r['estimated_cost_usd'] for r in sol_results),
    'estimated_new_request_cost_usd': sum(r['estimated_cost_usd'] for r in sol_results if not r['cached']),
    'observation_counts': dict(Counter(r['observation_verdict'] for r in sol_checks)) if sol_content_ready else None,
    'triage_decision_agreement_count': sum(r['decision_matches_reference'] is True for r in sol_checks) if sol_content_ready else None,
    'content_compatible_cases': sum(r['grounded'] is True for r in sol_checks) if sol_content_ready else None,
    'content_gate_passed': sol_gate, 'comparison': sol_comparison,
    'manual_csv_unchanged': True, 'manual_csv_sha256': SOL_MANUAL_HASH,
    'source_unchanged': True, 'source_sha256': SOL_SOURCE_HASHES, 'repair_applied': False,
    'reference': str(SOL_REF_PATH.relative_to(SOL_REPO)), 'reference_sha256': file_sha256(SOL_REF_PATH),
    'paid_api_run_id': SOL_REFERENCE.get('api_run_id'),
    'receipt_first_started_at': min((r['started_at'] for r in sol_results), default=None),
    'receipt_last_finished_at': max((r['finished_at'] for r in sol_results), default=None),
    'remaining_requests': 0, 'reviewed_unique_photos_in_both_pilots': 20 if len(sol_results) == 10 else 10+len(sol_results)}
sol_cards = []
sol_geometry_by_key = {tuple(r['case_key']): r for r in SOL_GEOMETRY}
for receipt, check in zip(sol_results, sol_checks):
    key = tuple(receipt['case_key']); geometry = sol_geometry_by_key[key]; ref = sol_reference_by_key[key]; review = receipt['review']
    title = f'Task nguồn {key[0]} / ann {key[1]} — {review["decision"]} — {check["observation_verdict"] or "PENDING"}'
    photos = ''.join(f'<a href="{html.escape(geometry[k])}"><img src="{html.escape(geometry[k])}" alt="{k}"></a>' for k in ['native', 'cutout', 'edge_zoom'])
    preview_rel = os.path.relpath(SOL_REPO / geometry['preview'], SOL_ARTIFACT)
    obs = ''.join('<li>'+html.escape(s)+'</li>' for s in review['observations'])
    sol_cards.append('<details open><summary>'+html.escape(title)+'</summary><p><b>Kỳ vọng trước API:</b> '+html.escape(ref['visual_expectation'])+'</p>'
        '<p><b>Kiểm tra phản hồi:</b> '+html.escape(check['notes'])+'</p><p>Chênh cạnh L/T/R/B (px): '+str(geometry['edge_gaps_ltrb_px'])+'</p>'
        '<div class="photos">'+photos+'</div><p><a href="'+html.escape(preview_rel)+'">Card nguyên bản có bbox/polygon</a></p>'
        '<p><b>Lý do Sol:</b> '+html.escape(review['reason'])+'</p><ul>'+obs+'</ul><p>needs_human='+str(review['needs_human'])+'; confidence tự báo='+str(review['confidence'])+'</p></details>')
(SOL_ARTIFACT / 'content_report.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Sol pilot new 10</title>'
    '<style>body{font:16px system-ui;max-width:1300px;margin:24px auto;padding:0 16px}details{border:1px solid #ccc;padding:14px;margin:16px 0}'
    'summary{font-weight:600;cursor:pointer}.photos{display:flex;gap:12px;align-items:flex-start}.photos a{width:33%}.photos img{max-width:100%;max-height:650px}'
    '@media(max-width:700px){.photos{display:block}.photos a{width:100%}}</style>'
    '<h1>GPT-6.1 Sol — 10 ảnh mới</h1><p>'+html.escape(sol_run['status'])+'; '+str(sol_summary['content_compatible_cases'])+'/10 phù hợp rubric. '
    'Luna trước: 7/10 trên tập khác. Không phải so sánh cùng ảnh hoặc accuracy nhãn; chưa sửa nhãn.</p>'+''.join(sol_cards)+'</html>')
sol_summary['html_report'] = str((SOL_ARTIFACT / 'content_report.html').relative_to(SOL_REPO))
write_json(SOL_ARTIFACT / 'run.json', sol_run)
write_json(SOL_ARTIFACT / 'case_content_checks.json', sol_checks)
write_json(SOL_ARTIFACT / 'quality_audit.json', sol_audit)
write_json(SOL_ARTIFACT / 'reference_snapshot.json', SOL_REFERENCE)
write_json(SOL_ARTIFACT / 'comparison.json', sol_comparison)
write_json(SOL_ARTIFACT / 'summary.json', sol_summary)
if sol_results:
    write_json(SOL_OUT / 'latest_summary.json', sol_summary)
print(json.dumps(sol_summary, ensure_ascii=False, indent=2))
print('Report:', sol_summary['html_report'])
for check in sol_checks:
    print(check['case_key'], check['decision'], check['observation_verdict'], 'grounded=', check['grounded'], check['notes'])


{
  "run": {
    "run_id": "11cf4169-bd69-4331-9726-768c48afd9ef",
    "purpose": "Sol pilot on 10 photos excluded from Luna, then content verification",
    "started_at": "2026-10-07T23:11:57+07:00",
    "finished_at": "2026-10-07T23:11:59+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "git_dirty": true,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-bbox-polygon-review-v3",
      "endpoint": "https://api.openai.com/v1/responses",
      "reasoning_effort": "low",
      "image_detail": "high",
      "cutout_max_side": 512,
      "max_output_tokens": 3000,
      "pilot_cases": 10,
      "workers": 2,
      "exclude_previous_model": "gpt-6-luna",
      "require_visual_pilot_audit": true,
      "apply_label_repairs": false,
      "write_manual_decisions": false,
      "pricing_checked_on": "2026-10-07",
      "pricing_mode": "standard_short_context",
     

## 13. Batch API cho phần còn lại — 08/10/2026

Theo yêu cầu mới, chạy phần còn lại bằng **gpt-6.1-sol Batch API**, cùng prompt v3/low và hai ảnh/case. Queue thực tế còn 202 ảnh chưa gửi AI sau hai pilot 10 ảnh. Gửi **đúng 200 ảnh** theo yêu cầu; **2 ảnh deferred** được ghi rõ trong config và output. Bộ duyệt tự động từ chối phạm vi 202 trước upload; bản chuẩn bị 202 ảnh giữ lại như preflight không gửi. Chi phí Batch bằng 50% Standard, cửa sổ xử lý tối đa 24h theo [OpenAI Docs](https://developers.openai.com/api/docs/guides/batch). [Cấu hình](../../configs/data/annotation_review_sol_batch_v1.json).

Đây là yêu cầu mở rộng **đề xuất** trực tiếp của người dùng ngày 08/10, dù pilot nội dung FAIL. Gate pilot được giữ nguyên; không tự coi tất cả kết quả là nhãn đúng. Nhãn CSV và raw data chỉ đọc. Nghiệm thu đủ request/IDs/schema/model/usage/provenance bằng cell; nghiệm thu pixel ownership và sửa nhãn vẫn cần review/mask.

Các cell phần 13–16 chạy độc lập với phần CUDA. Đặt `BATCH_SUBMIT=True` hoặc `AUTOCHECKOUT_REVIEW_BATCH_SUBMIT=1` để upload/create; `BATCH_REFRESH=True` hoặc `AUTOCHECKOUT_REVIEW_BATCH_REFRESH=1` để poll/tải kết quả. Mặc định hai cờ tắt. File state lưu file ID/batch ID/create intent, chạy lại không tạo thêm batch. Nếu mất response create, đối chiếu metadata remote trước khi tiếp tục để tránh trả phí hai lần.

Đặt `BATCH_WAIT_SECONDS` (mặc định 0) để poll 30 giây/lần trong một cell; chạy riêng cell theo dõi để tiếp tục sau ngắt phiên. JSONL kèm ảnh inline được chia nhỏ dưới 190 MB, kiểm tra SHA-256 trước upload. Input/cache tại `data/processed/synthetic_annotation_review_v1/ai/gpt-6.1-sol_batch_200_v1/`, log/HTML/acceptance theo run ID ở `runs/data_preprocessing/sol_batch/`.


In [1]:
import os, sys, json, uuid, platform, subprocess, time, csv, html, hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter
SB_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pdp/main.py').exists())
if str(SB_REPO) not in sys.path:
    sys.path.insert(0, str(SB_REPO))
from tools.ai_annotation_review import PROMPT, PROMPT_VERSION, file_sha256, write_json, load_api_key, timestamp
from tools.ai_annotation_batch import MODEL as SB_MODEL, TERMINAL, BatchClient, prepare_bundle, submit_bundle, poll_bundle, parse_outputs
BATCH_SUBMIT = os.getenv('AUTOCHECKOUT_REVIEW_BATCH_SUBMIT', '0') == '1'  # Có thể đổi thành True.
BATCH_REFRESH = os.getenv('AUTOCHECKOUT_REVIEW_BATCH_REFRESH', '0') == '1'  # Poll/tải, không tạo batch.
BATCH_WAIT_SECONDS = int(os.getenv('AUTOCHECKOUT_REVIEW_BATCH_WAIT_SECONDS', '0'))
SB_RAW = Path(os.getenv('AUTOCHECKOUT_DATA_ROOT', str(SB_REPO/'data/archive'))).resolve()
SB_REVIEW_ROOT = SB_REPO/'data/processed/synthetic_annotation_review_v1'
SB_OUT = SB_REVIEW_ROOT/'ai/gpt-6.1-sol_batch_200_v1'
SB_OUT.mkdir(parents=True, exist_ok=True)
SB_CONFIG_PATH = SB_REPO/'configs/data/annotation_review_sol_batch_v1.json'
SB_SETTINGS = json.loads(SB_CONFIG_PATH.read_text())
assert SB_SETTINGS['model'] == SB_MODEL == 'gpt-6.1-sol' and SB_SETTINGS['prompt_version'] == PROMPT_VERSION
assert SB_SETTINGS['authorization']['suggestions_authorized_despite_failed_pilot'] is True
assert SB_SETTINGS['reasoning_effort'] == 'low' and SB_SETTINGS['max_output_tokens'] == 3000
SB_ARTIFACT = SB_REPO/'runs/data_preprocessing/sol_batch'/str(uuid.uuid4())
SB_ARTIFACT.mkdir(parents=True, exist_ok=True)
SB_RUN = {'run_id': SB_ARTIFACT.name, 'purpose': 'Remaining annotation suggestions via Batch API, with notebook acceptance',
    'started_at': timestamp(), 'finished_at': None, 'timezone': 'Asia/Bangkok', 'host': platform.node(),
    'python': sys.version.split()[0], 'cuda_executed': False, 'settings': SB_SETTINGS,
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()),
    'code_sha256': file_sha256(SB_REPO/'tools/ai_annotation_batch.py'), 'status': 'PREPARING'}
SB_QUEUE = {(r['source_task'], r['annotation_id']): r for r in json.loads((SB_REVIEW_ROOT/'review_queue.json').read_text())}
SB_VISUAL = json.loads((SB_REVIEW_ROOT/'review_summary.json').read_text())
assert SB_VISUAL['visual_acceptance'] == 'PASS'
SB_ALL = json.loads((SB_REPO/SB_VISUAL['visual_manifest']).read_text())
assert len(SB_ALL) == len(SB_QUEUE) == 222
for r in SB_ALL:
    q = SB_QUEUE[r['source_task'], r['annotation_id']]
    assert q['source_sha256'] == r['source_sha256']
    r.update({k: q[k] for k in ['bbox', 'polygon_bbox']})
SB_PILOT_FILES = [SB_REVIEW_ROOT/'ai/gpt-6-luna_v1/pilot_results.json', SB_REVIEW_ROOT/'ai/gpt-6.1-sol_pilot_v1/pilot_results.json']
SB_EXCLUDED_KEYS = {tuple(r['case_key']) for path in SB_PILOT_FILES for r in json.loads(path.read_text())}
SB_EXCLUDED_IMAGES = {SB_QUEUE[key]['image_file'] for key in SB_EXCLUDED_KEYS}
assert len(SB_EXCLUDED_KEYS) == len(SB_EXCLUDED_IMAGES) == 20
SB_ELIGIBLE = sorted([r for r in SB_ALL if (r['source_task'], r['annotation_id']) not in SB_EXCLUDED_KEYS
                     and r['image_file'] not in SB_EXCLUDED_IMAGES], key=lambda r: (r['source_task'], r['annotation_id']))
SB_SELECTED = SB_ELIGIBLE[:SB_SETTINGS['target_cases']]
SB_DEFERRED = SB_ELIGIBLE[SB_SETTINGS['target_cases']:]
assert len(SB_ELIGIBLE) == 202 and len(SB_SELECTED) == SB_SETTINGS['target_cases']
assert len({r['image_file'] for r in SB_SELECTED}) == len(SB_SELECTED)
SB_MANUAL_HASH = file_sha256(SB_REVIEW_ROOT/'review_decisions.csv')
SB_SOURCE_HASHES = {str(t): file_sha256(SB_RAW/f'synth/group/task{t}/annotations.json') for t in range(1, 5)}
assert SB_SOURCE_HASHES == SB_VISUAL['source_sha256']
SB_VISUAL_ASSETS = json.loads((SB_REPO/Path(SB_VISUAL['visual_manifest']).parent/'source_asset_sha256.json').read_text())
SB_INPUT_HASHES = {'raw': {}, 'previews': {}}
for case in SB_SELECTED:
    for name in [case['image_file'], case['cutout']]:
        SB_INPUT_HASHES['raw'][name] = file_sha256(SB_RAW/name)
        assert SB_INPUT_HASHES['raw'][name] == SB_VISUAL_ASSETS[name]
    SB_INPUT_HASHES['previews'][case['preview']] = file_sha256(SB_REPO/case['preview'])
    assert SB_INPUT_HASHES['previews'][case['preview']] == case['preview_sha256']
SB_MANIFEST = prepare_bundle(SB_SELECTED, repo=SB_REPO, raw_root=SB_RAW, out_dir=SB_OUT,
    max_bytes=SB_SETTINGS['max_jsonl_bytes'], max_requests=SB_SETTINGS['max_requests_per_shard'])
SB_RUN.update(status='INPUT_ACCEPTANCE_PASS', selected_cases=len(SB_SELECTED), remaining_before=len(SB_ELIGIBLE),
    deferred_cases=len(SB_DEFERRED), manifest_fingerprint=SB_MANIFEST['fingerprint'])
write_json(SB_ARTIFACT/'run.json', SB_RUN)
write_json(SB_ARTIFACT/'selection.json', {'selected': SB_SELECTED, 'excluded_pilots': sorted(SB_EXCLUDED_KEYS), 'deferred': SB_DEFERRED})
write_json(SB_ARTIFACT/'source_asset_sha256.json', SB_INPUT_HASHES)
write_json(SB_ARTIFACT/'input_acceptance.json', {'passed': True, 'checked_at': timestamp(), 'expected_cases': len(SB_SELECTED),
    'pilot_photo_overlap': 0, 'source_sha256': SB_SOURCE_HASHES, 'manual_csv_sha256': SB_MANUAL_HASH,
    'manifest_fingerprint': SB_MANIFEST['fingerprint'], 'shards': SB_MANIFEST['shards']})
(SB_ARTIFACT/'prompt.txt').write_text(PROMPT)
print(json.dumps({'run': SB_RUN, 'input_acceptance': 'PASS', 'shards': SB_MANIFEST['shards'],
    'estimated_cost_from_pilot_usd': .0837/10*len(SB_SELECTED)*.5}, ensure_ascii=False, indent=2))


{
  "run": {
    "run_id": "4cb8fadd-91fa-4184-8954-1406a53cf12a",
    "purpose": "Remaining annotation suggestions via Batch API, with notebook acceptance",
    "started_at": "2026-10-08T00:34:19+07:00",
    "finished_at": null,
    "timezone": "Asia/Bangkok",
    "host": "Laptop",
    "python": "3.14.4",
    "cuda_executed": false,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-bbox-polygon-review-v3",
      "endpoint": "/v1/responses",
      "completion_window": "24h",
      "pricing_mode": "batch_short_context",
      "pricing_multiplier_vs_standard": 0.5,
      "usd_per_million_tokens": {
        "input": 1.0,
        "cached_input": 0.05,
        "cache_write": 1.25,
        "output": 5.0
      },
      "reasoning_effort": "low",
      "image_detail": "high",
      "cutout_max_side": 512,
      "max_output_tokens": 3000,
      "target_cases": 200,
      "selection": "Exactly 200 remaining photos ordered by source_task/annotation_id; exclude both pilo

## 14. Upload/create, theo dõi trạng thái và tải receipt

Mỗi shard chỉ được tạo một lần. Chỉ tải file output/error khi batch terminal; đối chiếu `custom_id` thay vì thứ tự dòng. `failed`, `expired`, `cancelled` được ghi nhận thật, không xem là xử lý đủ và không tự gửi lại các ảnh lỗi. Chi phí của response incomplete vẫn được giữ. Khi hết thời gian chờ nhưng batch chưa terminal, notebook ghi BATCH_PENDING; chạy lại cell để tiếp tục.


In [2]:
SB_SUBMISSION_PATH = SB_OUT/'submission.json'
SB_STATE = json.loads(SB_SUBMISSION_PATH.read_text()) if SB_SUBMISSION_PATH.exists() else None
if BATCH_SUBMIT or BATCH_REFRESH:
    SB_CLIENT = BatchClient(load_api_key(SB_REPO/'.env'))
    if BATCH_SUBMIT:
        SB_STATE = submit_bundle(SB_MANIFEST, out_dir=SB_OUT, client=SB_CLIENT)
    if SB_STATE:
        sb_deadline = time.monotonic() + max(0, BATCH_WAIT_SECONDS)
        while True:
            SB_STATE = poll_bundle(SB_MANIFEST, out_dir=SB_OUT, client=SB_CLIENT)
            print(timestamp(), [(s['batch_id'], s['batch']['status'], s['batch'].get('request_counts')) for s in SB_STATE['shards']])
            write_json(SB_ARTIFACT/'submission_snapshot.json', SB_STATE)
            if SB_STATE['terminal'] or time.monotonic() >= sb_deadline:
                break
            time.sleep(min(SB_SETTINGS['poll_interval_seconds'], max(0, sb_deadline-time.monotonic())))
else:
    print('Network disabled; using saved Batch state/output files only. No upload/create/poll calls.')
SB_RUN['batch_ids'] = [s['batch_id'] for s in SB_STATE['shards']] if SB_STATE else []
SB_RUN['status'] = ('BATCH_TERMINAL' if SB_STATE.get('terminal') else 'BATCH_PENDING') if SB_STATE else 'INPUT_READY_NOT_SUBMITTED'
write_json(SB_ARTIFACT/'run.json', SB_RUN)
print(json.dumps({'batch_ids': SB_RUN['batch_ids'], 'status': SB_RUN['status']}, ensure_ascii=False, indent=2))


Network disabled; using saved Batch state/output files only. No upload/create/poll calls.
{
  "batch_ids": [
    "batch_6ac67e913d288190aa1dae5115275f17"
  ],
  "status": "BATCH_TERMINAL"
}


## 15. Nghiệm thu toàn bộ response và tổng hợp đề xuất

Các gate cơ học gồm: đủ `custom_id` duy nhất theo manifest, đúng model/IDs/schema, response completed, không có request error/missing, hashes nguồn/input không đổi, CSV manual không đổi và chi phí Batch được tính từ usage gồm cache-write/reasoning. Không nhận response lỗi vào danh sách đã parse; không bỏ qua phần đã bị tính tiền. Gate API PASS chứng minh workflow nhận đủ đề xuất có cấu trúc, không chứng minh nội dung nhãn đúng.

HTML tổng hợp lưu card, cutout, đề xuất, lý do, observations và cờ cần người review. Tất cả đề xuất giữ/tái sinh vẫn cần review trước khi nghiệm thu labels vì pilot nội dung FAIL. Chỉ số confidence tự báo không được coi là accuracy.


In [3]:
SB_ACCEPTED = []
SB_REJECTED = []
SB_API_PASS = False
SB_RAW_OUTPUTS = []
if SB_STATE and SB_STATE.get('terminal'):
    for shard in SB_STATE['shards']:
        for name in ['output', 'error']:
            path = shard.get(name+'_path')
            if path:
                assert file_sha256(Path(path)) == shard[name+'_sha256']
                SB_RAW_OUTPUTS.extend(json.loads(line) for line in Path(path).read_text().splitlines() if line.strip())
    SB_ACCEPTED, SB_REJECTED = parse_outputs(SB_MANIFEST, SB_RAW_OUTPUTS, batch_ids=SB_RUN['batch_ids'])
    SB_API_PASS = (len(SB_ACCEPTED) == len(SB_SELECTED) and not SB_REJECTED
                   and len(SB_RAW_OUTPUTS) == len(SB_SELECTED)
                   and all(s['batch']['status'] == 'completed' for s in SB_STATE['shards']))
    write_json(SB_OUT/'accepted_results.json', SB_ACCEPTED)
    write_json(SB_OUT/'rejected_results.json', SB_REJECTED)
    write_json(SB_ARTIFACT/'accepted_results.json', SB_ACCEPTED)
    write_json(SB_ARTIFACT/'rejected_results.json', SB_REJECTED)
    for receipt in SB_ACCEPTED+SB_REJECTED:
        write_json(SB_OUT/'cases'/(receipt['custom_id']+'.json'), receipt)
    SB_RUN['status'] = 'API_ACCEPTANCE_PASS_CONTENT_PENDING' if SB_API_PASS else 'API_ACCEPTANCE_FAIL'
else:
    print('Batch not terminal; result/schema/coverage acceptance is PENDING, not PASS.')
for task, digest in SB_SOURCE_HASHES.items():
    assert file_sha256(SB_RAW/f'synth/group/task{task}/annotations.json') == digest
for name, digest in SB_INPUT_HASHES['raw'].items():
    assert file_sha256(SB_RAW/name) == digest
for name, digest in SB_INPUT_HASHES['previews'].items():
    assert file_sha256(SB_REPO/name) == digest
assert file_sha256(SB_REVIEW_ROOT/'review_decisions.csv') == SB_MANUAL_HASH
for shard in SB_MANIFEST['shards']:
    assert file_sha256(Path(shard['path'])) == shard['sha256']
SB_RECEIPTS = SB_ACCEPTED + SB_REJECTED
SB_COST = sum(r['estimated_cost_usd'] or 0 for r in SB_RECEIPTS)
SB_USAGE = {'input_tokens': sum((r['usage'] or {}).get('input_tokens', 0) for r in SB_RECEIPTS),
    'cached_input_tokens': sum(((r['usage'] or {}).get('input_tokens_details') or {}).get('cached_tokens', 0) for r in SB_RECEIPTS),
    'cache_write_tokens': sum(((r['usage'] or {}).get('input_tokens_details') or {}).get('cache_write_tokens', 0) for r in SB_RECEIPTS),
    'output_tokens': sum((r['usage'] or {}).get('output_tokens', 0) for r in SB_RECEIPTS),
    'reasoning_tokens': sum(((r['usage'] or {}).get('output_tokens_details') or {}).get('reasoning_tokens', 0) for r in SB_RECEIPTS)}
SB_SUMMARY = {'run': SB_RUN, 'model': SB_MODEL, 'batch_ids': SB_RUN['batch_ids'],
    'expected_cases': len(SB_SELECTED), 'valid_results': len(SB_ACCEPTED), 'rejected_cases': len(SB_REJECTED),
    'raw_output_rows': len(SB_RAW_OUTPUTS), 'api_acceptance': 'PASS' if SB_API_PASS else ('FAIL' if SB_STATE and SB_STATE.get('terminal') else 'PENDING'),
    'decision_counts': dict(Counter(r['review']['decision'] for r in SB_ACCEPTED)),
    'model_needs_human_count': sum(r['review']['needs_human'] for r in SB_ACCEPTED),
    'all_suggestions_require_review': True, 'pilot_content_gate': 'FAIL_PRESERVED',
    'content_acceptance': 'PENDING_VISUAL_REVIEW', 'ground_truth_accuracy': None,
    'label_repairs_accepted': 0, 'manual_decisions_written': 0, 'usage': SB_USAGE,
    'estimated_batch_cost_usd': SB_COST, 'cost_is_billing_invoice': False,
    'cost_estimate_complete': all(not r['usage'] or r['estimated_cost_usd'] is not None for r in SB_RECEIPTS),
    'source_and_manual_csv_unchanged': True, 'source_sha256': SB_SOURCE_HASHES, 'manual_csv_sha256': SB_MANUAL_HASH,
    'deferred_cases': [(r['source_task'], r['annotation_id']) for r in SB_DEFERRED],
    'accepted_results': str((SB_ARTIFACT/'accepted_results.json').relative_to(SB_REPO)),
    'manifest': str((SB_OUT/'manifest.json').relative_to(SB_REPO))}
SB_GATES = {'input_unique_no_pilot_overlap': True, 'source_assets_unchanged': True,
    'manual_csv_unchanged': True, 'frozen_jsonl_unchanged': True,
    'all_expected_outputs_completed_model_ids_schema': SB_API_PASS if SB_STATE and SB_STATE.get('terminal') else None,
    'no_failed_or_missing_requests': not SB_REJECTED if SB_STATE and SB_STATE.get('terminal') else None,
    'content_review_all_cases': None, 'labels_accepted': False}
write_json(SB_ARTIFACT/'acceptance_gates.json', SB_GATES)
write_json(SB_ARTIFACT/'summary.json', SB_SUMMARY)
write_json(SB_OUT/'latest_summary.json', SB_SUMMARY)
if SB_RECEIPTS:
    with (SB_ARTIFACT/'suggestions.csv').open('w', newline='') as target:
        writer = csv.DictWriter(target, fieldnames=['source_task','annotation_id','decision','confidence','model_needs_human','review_required','reason'])
        writer.writeheader()
        for receipt in SB_ACCEPTED:
            r = receipt['review']
            writer.writerow({'source_task': r['source_task'], 'annotation_id': r['annotation_id'], 'decision': r['decision'],
                'confidence': r['confidence'], 'model_needs_human': r['needs_human'], 'review_required': True, 'reason': r['reason']})
    write_json(SB_OUT/'experiment_ledger.json', {'batch_ids': SB_RUN['batch_ids'], 'pricing_mode': 'batch',
        'unique_response_ids': sorted({r['response_id'] for r in SB_RECEIPTS if r['response_id']}),
        'saved_receipts': len(SB_RECEIPTS), 'completed_accepted': len(SB_ACCEPTED), 'rejected': len(SB_REJECTED),
        'usage': SB_USAGE, 'estimated_cost_usd': SB_COST, 'notebook_run_id': SB_ARTIFACT.name,
        'reparse_creates_no_new_requests': True})
print(json.dumps({'summary': SB_SUMMARY, 'acceptance_gates': SB_GATES}, ensure_ascii=False, indent=2))


{
  "summary": {
    "run": {
      "run_id": "4cb8fadd-91fa-4184-8954-1406a53cf12a",
      "purpose": "Remaining annotation suggestions via Batch API, with notebook acceptance",
      "started_at": "2026-10-08T00:34:19+07:00",
      "finished_at": null,
      "timezone": "Asia/Bangkok",
      "host": "Laptop",
      "python": "3.14.4",
      "cuda_executed": false,
      "settings": {
        "model": "gpt-6.1-sol",
        "prompt_version": "rpc-bbox-polygon-review-v3",
        "endpoint": "/v1/responses",
        "completion_window": "24h",
        "pricing_mode": "batch_short_context",
        "pricing_multiplier_vs_standard": 0.5,
        "usd_per_million_tokens": {
          "input": 1.0,
          "cached_input": 0.05,
          "cache_write": 1.25,
          "output": 5.0
        },
        "reasoning_effort": "low",
        "image_detail": "high",
        "cutout_max_side": 512,
        "max_output_tokens": 3000,
        "target_cases": 200,
        "selection": "Exactly 200 r

## 16. Báo cáo review, kiểm tra nội dung mẫu và kết thúc run

Cell cuối xuất HTML cho toàn bộ đề xuất đã parse, chọn mẫu theo task/decision và mức lệch để kiểm tra trực quan. Reference mẫu (nếu có) phải khóa đúng output/source fingerprints và ghi notes; không dùng lại audit pilot cho output batch. Kiểm tra mẫu không thay thế nghiệm thu nội dung tất cả case. Nếu còn kết luận chưa rõ, giữ gate nội dung pending/fail và nhãn chưa nghiệm thu.


In [4]:
sb_cards = []
SB_BY_KEY = {(r['source_task'],r['annotation_id']): r for r in SB_SELECTED}
for receipt in SB_ACCEPTED:
    r = receipt['review']; key = tuple(receipt['case_key']); case = SB_BY_KEY[key]
    preview = os.path.relpath(SB_REPO/case['preview'], SB_ARTIFACT)
    cutout = os.path.relpath(SB_RAW/case['cutout'], SB_ARTIFACT)
    observations = ''.join('<li>'+html.escape(s)+'</li>' for s in r['observations'])
    title = f"Task nguồn {key[0]} / ann {key[1]} — {r['decision']} — IoU {case['iou']:.4f}"
    sb_cards.append('<details data-decision="'+html.escape(r['decision'])+'"><summary>'+html.escape(title)+'</summary>'
        '<p>Model needs_human='+str(r['needs_human'])+'; confidence tự báo='+str(r['confidence'])+'; mọi đề xuất vẫn cần review.</p>'
        '<div class="photos"><a href="'+html.escape(preview)+'"><img loading="lazy" src="'+html.escape(preview)+'" alt="Native comparison card"></a>'
        '<a href="'+html.escape(cutout)+'"><img loading="lazy" src="'+html.escape(cutout)+'" alt="Original cutout"></a></div>'
        '<p>'+html.escape(r['reason'])+'</p><ul>'+observations+'</ul></details>')
(SB_ARTIFACT/'content_report.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Sol Batch review</title>'
    '<style>body{font:16px system-ui;max-width:1450px;margin:24px auto;padding:0 16px}details{border:1px solid #ccc;padding:12px;margin:12px 0}'
    'summary{font-weight:600;cursor:pointer}.photos{display:flex;gap:12px;align-items:flex-start}.photos a:first-child{width:78%}.photos a:last-child{width:22%}'
    '.photos img{max-width:100%;max-height:800px}@media(max-width:700px){.photos{display:block}.photos a{width:100%!important}}</style>'
    '<h1>GPT-6.1 Sol Batch — '+str(len(SB_ACCEPTED))+'/'+str(len(SB_SELECTED))+' đề xuất</h1>'
    '<p>API acceptance: '+SB_SUMMARY['api_acceptance']+'. Pilot nội dung FAIL; mọi đề xuất cần review, chưa sửa nhãn.</p>'
    '<label>Lọc: <select id="filter"><option value="all">Tất cả</option><option>regenerate</option><option>pending</option><option>keep_with_justification</option></select></label>'
    + ''.join(sb_cards)+'<script>document.querySelector("#filter").onchange=e=>document.querySelectorAll("details").forEach(d=>d.hidden=e.target.value!=="all"&&d.dataset.decision!==e.target.value)</script></html>')
# Deterministic visual sample: source-task/decision groups plus largest geometry discrepancies.
sb_sample = []; sb_seen = set()
sb_ordered = sorted(SB_ACCEPTED, key=lambda r: (SB_BY_KEY[tuple(r['case_key'])]['iou'], r['case_key']))
sb_groups = sorted({(r['case_key'][0],r['review']['decision']) for r in sb_ordered})
sb_candidates = []
for task, decision in sb_groups:
    group = [r for r in sb_ordered if r['case_key'][0]==task and r['review']['decision']==decision]
    sb_candidates.append(group[0])
for task, decision in sb_groups:
    group = [r for r in sb_ordered if r['case_key'][0]==task and r['review']['decision']==decision]
    sb_candidates.append(group[-1])
for receipt in sb_candidates:
    if receipt['fingerprint'] not in sb_seen and len(sb_sample)<12:
        sb_sample.append(receipt); sb_seen.add(receipt['fingerprint'])
for receipt in sb_ordered:
    if len(sb_sample)==12: break
    if receipt['fingerprint'] not in sb_seen:
        sb_sample.append(receipt); sb_seen.add(receipt['fingerprint'])
write_json(SB_ARTIFACT/'content_sample_selection.json', [{'case_key': r['case_key'], 'fingerprint': r['fingerprint'],
    'review_sha256': hashlib.sha256(json.dumps(r['review'], ensure_ascii=False, sort_keys=True).encode()).hexdigest(),
    'decision': r['review']['decision'], 'source_sha256': r['source_sha256']} for r in sb_sample])
SB_SAMPLE_REFERENCE = SB_REPO/'configs/data/annotation_review_sol_batch_sample_reference_v1.json'
SB_SAMPLE_STATUS = 'PENDING'
if SB_SAMPLE_REFERENCE.exists() and sb_sample:
    sb_ref = json.loads(SB_SAMPLE_REFERENCE.read_text())
    sb_expected = {r['fingerprint']:r for r in sb_sample}
    assert sb_ref['manifest_fingerprint'] == SB_MANIFEST['fingerprint']
    assert {r['fingerprint'] for r in sb_ref['case_audits']} == set(sb_expected)
    for check in sb_ref['case_audits']:
        r = sb_expected[check['fingerprint']]
        assert check['case_key'] == r['case_key']
        case = SB_BY_KEY[tuple(r['case_key'])]
        assert check['image_sha256'] == SB_INPUT_HASHES['raw'][case['image_file']]
        assert check['review_sha256'] == hashlib.sha256(json.dumps(r['review'], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
        assert check['source_sha256'] == r['source_sha256'] and len(check['notes'].strip()) >= 30
        assert type(check['grounded']) is bool
    SB_SAMPLE_STATUS = 'PASS' if all(r['grounded'] for r in sb_ref['case_audits']) else 'FAIL'
    write_json(SB_ARTIFACT/'content_sample_reference.json', sb_ref)
    SB_SUMMARY['content_sample_grounded'] = sum(r['grounded'] for r in sb_ref['case_audits'])
    SB_SUMMARY['content_sample_reference'] = str(SB_SAMPLE_REFERENCE.relative_to(SB_REPO))
    SB_SUMMARY['content_sample_reference_sha256'] = file_sha256(SB_SAMPLE_REFERENCE)
    sb_sample_cards = []
    for check in sb_ref['case_audits']:
        title = str(check['case_key']) + ' — ' + check['observation_verdict']
        photos = ''.join('<a href="'+html.escape(os.path.relpath(SB_REPO/check[name],SB_ARTIFACT))+'"><img src="'+html.escape(os.path.relpath(SB_REPO/check[name],SB_ARTIFACT))+'" alt="'+name+'"></a>' for name in ['native','edge_zoom'])
        comparison = html.escape(os.path.relpath(SB_REPO/check['comparison_with_cutout'],SB_ARTIFACT))
        sb_sample_cards.append('<details open><summary>'+html.escape(title)+'</summary><p>'+html.escape(check['notes'])+'</p><div class="photos">'+photos+'</div><p><a href="'+comparison+'">Card đối chiếu và cutout</a></p></details>')
    (SB_ARTIFACT/'content_sample_report.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Batch content sample</title><style>body{font:16px system-ui;max-width:1200px;margin:24px auto;padding:0 16px}details{border:1px solid #ccc;padding:12px;margin:12px 0}.photos{display:flex;gap:12px}.photos a{width:50%}.photos img{max-width:100%;max-height:600px}</style><h1>Mẫu 12 case — '+str(SB_SUMMARY['content_sample_grounded'])+'/12 phù hợp rubric</h1><p>Mẫu không ngẫu nhiên, không phải accuracy toàn bộ 200 nhãn. API/schema PASS; nội dung mẫu '+SB_SAMPLE_STATUS+'; nhãn chưa sửa.</p>'+''.join(sb_sample_cards)+'</html>')
    SB_SUMMARY['content_sample_report'] = str((SB_ARTIFACT/'content_sample_report.html').relative_to(SB_REPO))
    # Keep all-case gallery and content notes connected for the reviewer.
    sb_report_path = SB_ARTIFACT/'content_report.html'
    sb_report_html = sb_report_path.read_text()
    sb_report_html = sb_report_html.replace('<label>Lọc:', '<p><a href="content_sample_report.html">Đối chiếu nội dung mẫu 12 case: '+str(SB_SUMMARY['content_sample_grounded'])+'/12, '+SB_SAMPLE_STATUS+'</a></p><label>Lọc:')
    sb_report_path.write_text(sb_report_html)

SB_RUN['finished_at'] = timestamp()
SB_RUN['duration_seconds'] = (datetime.fromisoformat(SB_RUN['finished_at']) - datetime.fromisoformat(SB_RUN['started_at'])).total_seconds()
SB_SUMMARY.update(run=SB_RUN, content_sample_status=SB_SAMPLE_STATUS, content_sample_cases=len(sb_sample),
    html_report=str((SB_ARTIFACT/'content_report.html').relative_to(SB_REPO)),
    visual_sample_is_not_full_content_acceptance=True)
write_json(SB_ARTIFACT/'run.json', SB_RUN)
write_json(SB_ARTIFACT/'summary.json', SB_SUMMARY)
write_json(SB_OUT/'latest_summary.json', SB_SUMMARY)
print(json.dumps(SB_SUMMARY, ensure_ascii=False, indent=2))
print('HTML report:', SB_SUMMARY['html_report'])
print('Visual sample:', [r['case_key'] for r in sb_sample], 'sample status=', SB_SAMPLE_STATUS)
print('Manual labels remain separate; full content acceptance=', SB_SUMMARY['content_acceptance'])


{
  "run": {
    "run_id": "4cb8fadd-91fa-4184-8954-1406a53cf12a",
    "purpose": "Remaining annotation suggestions via Batch API, with notebook acceptance",
    "started_at": "2026-10-08T00:34:19+07:00",
    "finished_at": "2026-10-08T00:34:27+07:00",
    "timezone": "Asia/Bangkok",
    "host": "Laptop",
    "python": "3.14.4",
    "cuda_executed": false,
    "settings": {
      "model": "gpt-6.1-sol",
      "prompt_version": "rpc-bbox-polygon-review-v3",
      "endpoint": "/v1/responses",
      "completion_window": "24h",
      "pricing_mode": "batch_short_context",
      "pricing_multiplier_vs_standard": 0.5,
      "usd_per_million_tokens": {
        "input": 1.0,
        "cached_input": 0.05,
        "cache_write": 1.25,
        "output": 5.0
      },
      "reasoning_effort": "low",
      "image_detail": "high",
      "cutout_max_side": 512,
      "max_output_tokens": 3000,
      "target_cases": 200,
      "selection": "Exactly 200 remaining photos ordered by source_task/annotatio